# Main experiments: BO versus CMA-ES

Experimental development (4–6D), main 20-seed study (7), and targeted follow-ups (8A–8C). **Do not use Run all:** some cells take hours. Experiment 8A-R supersedes 8A and Experiment 8B-R supersedes 8B for final reporting. Results are saved separately in Drive; the original Colab-specific paths remain unchanged for historical reproducibility.

**Setup:** Google Colab with required Python packages and the project Google Drive folder. See the repository README for reproducibility instructions.


In [ ]:

!pip -q install cma scikit-learn scipy pandas matplotlib


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, json, platform, warnings
import cma, sklearn, scipy

from dataclasses import dataclass
from scipy.stats import qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

# ==========================================================
# CONFIGURATION
# ==========================================================

@dataclass(frozen=True)
class Config:
    d: int = 5
    budget: int = 200
    n_initial: int = 10
    low: float = -5.0
    high: float = 5.0
    acquisition_candidates: int = 512

CFG = Config()
SEEDS = [42, 123, 2026]
NOISE_FRACTIONS = [0.0, 0.05]

# ==========================================================
# BENCHMARK FUNCTIONS
# ==========================================================

def sphere(x):
    return float(np.sum(x**2))

def ackley(x):
    return float(
        -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        -np.exp(np.mean(np.cos(2*np.pi*x)))
        +20+np.e
    )

def rosenbrock(x):
    return float(np.sum(
        100*(x[1:]-x[:-1]**2)**2
        +(1-x[:-1])**2
    ))

def rastrigin(x):
    return float(
        10*len(x)+np.sum(
            x**2-10*np.cos(2*np.pi*x)
        )
    )

FUNCTIONS = {
    "Sphere": sphere,
    "Ackley": ackley,
    "Rosenbrock": rosenbrock,
    "Rastrigin": rastrigin
}

# Known global optimum value: 0 for all four.

# ==========================================================
# NOISE CALIBRATION
# ==========================================================

# Pilot convention: 5% of global objective IQR.
# Not a claim of equal local signal-to-noise ratio.
# Noise scales are calculated outside optimiser code.

scale_rng = np.random.default_rng(9001)
scale_X = scale_rng.uniform(
    CFG.low, CFG.high, (2000, CFG.d)
)

SCALES = {}

for name, func in FUNCTIONS.items():
    values = np.array([func(x) for x in scale_X])
    q25, q75 = np.percentile(values, [25, 75])
    SCALES[name] = float(q75-q25)

print("Objective IQR scales:")
print(SCALES)

# ==========================================================
# AUDITED EVALUATOR
# ==========================================================

class BudgetExceeded(Exception):
    pass

class Evaluator:
    def __init__(self, func, sigma, seed, budget):
        self._func = func
        self.sigma = float(sigma)
        self.budget = int(budget)
        self.rng = np.random.default_rng(seed)
        self.X = []
        self.y = []
        self.calls = 0

    def evaluate(self, x):
        if self.calls >= self.budget:
            raise BudgetExceeded("Evaluation budget exhausted")

        x = np.asarray(x, dtype=float).copy()

        assert x.shape == (CFG.d,)
        assert np.all(np.isfinite(x))
        assert np.all(x >= CFG.low-1e-8)
        assert np.all(x <= CFG.high+1e-8)

        # True value is private to the evaluator.
        true_value = self._func(x)
        observation = (
            true_value + self.rng.normal(0, self.sigma)
        )

        self.X.append(x)
        self.y.append(float(observation))
        self.calls += 1

        return float(observation)

# ==========================================================
# COMMON INITIAL DESIGN
# ==========================================================

def initial_design(seed):
    sampler = qmc.LatinHypercube(
        d=CFG.d, seed=seed
    )
    return qmc.scale(
        sampler.random(CFG.n_initial),
        CFG.low, CFG.high
    )

def initialise(evaluator, seed):
    X0 = initial_design(seed)
    y0 = np.array([
        evaluator.evaluate(x) for x in X0
    ])
    return X0, y0

# ==========================================================
# GAUSSIAN PROCESS MODEL
# ==========================================================

def fit_gp(X, y, sigma):
    Xs = (X-CFG.low)/(CFG.high-CFG.low)

    mean_y = float(np.mean(y))
    scale_y = max(float(np.std(y)), 1e-8)
    yn = (y-mean_y)/scale_y

    gp = GaussianProcessRegressor(
        kernel=Matern(
            length_scale=0.3,
            length_scale_bounds="fixed",
            nu=2.5
        ),
        alpha=max((sigma/scale_y)**2, 1e-8),
        normalize_y=False,
        optimizer=None
    )
    gp.fit(Xs, yn)
    return gp, mean_y, scale_y

# ==========================================================
# BAYESIAN OPTIMISATION
# ==========================================================

def run_bo(evaluator, seed):
    rng = np.random.default_rng(seed+7000)

    initialise(evaluator, seed)

    while evaluator.calls < evaluator.budget:
        X = np.asarray(evaluator.X)
        y = np.asarray(evaluator.y)

        gp, _, _ = fit_gp(
            X, y, evaluator.sigma
        )

        candidates = rng.uniform(
            0, 1,
            (CFG.acquisition_candidates, CFG.d)
        )

        mu, std = gp.predict(
            candidates, return_std=True
        )

        # Lower confidence bound for minimisation
        scores = mu - 2.0*std
        chosen = candidates[np.argmin(scores)]

        x_next = (
            CFG.low + chosen*(CFG.high-CFG.low)
        )
        evaluator.evaluate(x_next)

    X = np.asarray(evaluator.X)
    y = np.asarray(evaluator.y)

    gp, _, _ = fit_gp(X, y, evaluator.sigma)
    Xs = (X-CFG.low)/(CFG.high-CFG.low)

    posterior_mean = gp.predict(Xs)

    return {
        "observed_best_x":
            X[np.argmin(y)].copy(),
        "state_x":
            X[np.argmin(posterior_mean)].copy()
    }

# ==========================================================
# CMA-ES
# ==========================================================

def run_cma(evaluator, seed):
    X0, y0 = initialise(evaluator, seed)

    # CMA-ES uses the best noisy initial point
    # as its starting centre.
    x0 = X0[np.argmin(y0)].copy()

    es = cma.CMAEvolutionStrategy(
        x0, 2.0,
        {
            "seed": seed,
            "bounds": [CFG.low, CFG.high],
            "popsize": 8,
            "verbose": -9,
            "verb_disp": 0,
            "tolfun": 0,
            "tolx": 1e-14,
            "tolstagnation": 100000
        }
    )

    while evaluator.calls < evaluator.budget:
        candidates = es.ask()
        remaining = evaluator.budget-evaluator.calls
        batch = candidates[:remaining]

        values = [
            evaluator.evaluate(x)
            for x in batch
        ]

        if len(batch) == len(candidates):
            es.tell(candidates, values)
        else:
            # Final partial generation is charged
            # but not used for a distribution update.
            break

        if es.stop():
            # Preserve strict budget accounting:
            # restart from the current centre.
            centre = np.clip(
                es.mean, CFG.low, CFG.high
            )
            es = cma.CMAEvolutionStrategy(
                centre, 2.0,
                {
                    "seed": seed+evaluator.calls+1,
                    "bounds": [CFG.low, CFG.high],
                    "popsize": 8,
                    "verbose": -9,
                    "verb_disp": 0,
                    "tolfun": 0
                }
            )

    X = np.asarray(evaluator.X)
    y = np.asarray(evaluator.y)

    return {
        "observed_best_x":
            X[np.argmin(y)].copy(),
        "state_x":
            np.clip(
                np.asarray(es.mean),
                CFG.low, CFG.high
            ).copy()
    }

# ==========================================================
# INDEPENDENT SCORING
# ==========================================================

def execute_trial(fname, method, noise_fraction, seed):
    func = FUNCTIONS[fname]
    sigma = noise_fraction*SCALES[fname]

    evaluator = Evaluator(
        func=func,
        sigma=sigma,
        seed=seed+100000,
        budget=CFG.budget
    )

    start = time.perf_counter()

    if method == "BO":
        rec = run_bo(evaluator, seed)
    elif method == "CMA-ES":
        rec = run_cma(evaluator, seed)
    else:
        raise ValueError("Unknown method")

    elapsed = time.perf_counter()-start

    # Strict checks
    assert evaluator.calls == CFG.budget
    assert len(evaluator.X) == CFG.budget
    assert len(evaluator.y) == CFG.budget

    X = np.asarray(evaluator.X)
    assert np.all(np.isfinite(X))
    assert np.all(X >= CFG.low-1e-8)
    assert np.all(X <= CFG.high+1e-8)

    # Only scorer evaluates noise-free objective.
    observed_regret = func(rec["observed_best_x"])
    state_regret = func(rec["state_x"])

    # Oracle diagnostic, never fed to optimiser.
    oracle_regret = min(func(x) for x in X)

    record = {
        "function": fname,
        "method": method,
        "noise_fraction": noise_fraction,
        "sigma": sigma,
        "seed": seed,
        "budget": evaluator.calls,
        "observed_regret": observed_regret,
        "state_regret": state_regret,
        "oracle_regret": oracle_regret,
        "runtime_seconds": elapsed
    }

    trace = pd.DataFrame({
        "function": fname,
        "method": method,
        "noise_fraction": noise_fraction,
        "seed": seed,
        "evaluation": np.arange(1, CFG.budget+1),
        "observed_y": evaluator.y,
        **{
            f"x{j}": X[:,j]
            for j in range(CFG.d)
        }
    })

    return record, trace

print("FRAMEWORK DEFINED SUCCESSFULLY")


In [ ]:

# ==========================================================
# FRAMEWORK VALIDATION
# ==========================================================

print("RUNNING VALIDATION TESTS")

# Test 1: budget enforcement
e = Evaluator(sphere, 0.5, 42, 2)
e.evaluate(np.zeros(CFG.d))
e.evaluate(np.zeros(CFG.d))

try:
    e.evaluate(np.zeros(CFG.d))
    raise AssertionError("Budget check failed")
except BudgetExceeded:
    print("PASS: Budget enforcement")

# Test 2: deterministic noisy observations
a = Evaluator(sphere, 1.0, 123, 5)
b = Evaluator(sphere, 1.0, 123, 5)

ya = [a.evaluate(np.zeros(CFG.d)) for _ in range(5)]
yb = [b.evaluate(np.zeros(CFG.d)) for _ in range(5)]

assert np.array_equal(ya, yb)
assert len(set(ya)) > 1
print("PASS: Reproducible independent noise")

# Test 3: initial design reproducibility
assert np.array_equal(
    initial_design(42),
    initial_design(42)
)
print("PASS: Initial design reproducibility")

# Test 4: full optimiser accounting
for method in ["BO", "CMA-ES"]:
    r1, t1 = execute_trial(
        "Sphere", method, 0.05, 42
    )
    r2, t2 = execute_trial(
        "Sphere", method, 0.05, 42
    )

    assert r1["budget"] == CFG.budget
    assert np.allclose(
        t1["observed_y"],
        t2["observed_y"],
        rtol=0, atol=1e-8
    )
    assert np.isclose(
        r1["observed_regret"],
        r2["observed_regret"]
    )
    assert np.isclose(
        r1["state_regret"],
        r2["state_regret"]
    )

    print(f"PASS: {method} reproducibility and budget")

print("\nALL VALIDATION TESTS PASSED")


In [ ]:

# ==========================================================
# EXPERIMENT 4: MAIN FRAMEWORK PILOT
# ==========================================================

from pathlib import Path

OUT = Path("experiment_4")
OUT.mkdir(exist_ok=True)

records = []
traces = []

total = (
    len(FUNCTIONS)
    * 2
    * len(NOISE_FRACTIONS)
    * len(SEEDS)
)

print(f"STARTING {total} TRIALS")
print("=" * 75)

for fname in FUNCTIONS:
    for noise in NOISE_FRACTIONS:
        for seed in SEEDS:
            for method in ["BO", "CMA-ES"]:

                result, trace = execute_trial(
                    fname, method, noise, seed
                )

                records.append(result)
                traces.append(trace)

                print(
                    f"{len(records):02d}/{total} "
                    f"{fname:12s} "
                    f"{method:6s} "
                    f"noise={noise:.2f} "
                    f"seed={seed} "
                    f"regret={result['observed_regret']:.5f} "
                    f"state={result['state_regret']:.5f} "
                    f"time={result['runtime_seconds']:.2f}s",
                    flush=True
                )

                # Save after every successful trial
                pd.DataFrame(records).to_csv(
                    OUT / "results.csv", index=False
                )

df4 = pd.DataFrame(records)
trace4 = pd.concat(traces, ignore_index=True)

trace4.to_csv(
    OUT / "observations.csv", index=False
)

metadata = {
    "config": vars(CFG),
    "seeds": SEEDS,
    "noise_fractions": NOISE_FRACTIONS,
    "noise_scales": SCALES,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "cma": cma.__version__
}

with open(OUT / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nMEAN TRUE REGRET")
print("=" * 75)

summary = df4.groupby(
    ["function", "noise_fraction", "method"]
)[["observed_regret", "state_regret",
   "oracle_regret", "runtime_seconds"]].agg(
    ["mean", "std"]
).round(4)

print(summary.to_string())

print("\nBUDGET CHECK")
print(df4.groupby("method")["budget"].agg(
    ["min", "max", "count"]
))

assert len(df4) == total
assert (df4["budget"] == CFG.budget).all()

print("\nEXPERIMENT 4 COMPLETE")
print("Results saved in:", OUT)


In [ ]:

# ==========================================================
# EXPERIMENT 5: OPTIMISER VALIDATION
# BO baseline vs improved BO vs CMA-ES
# CPU-only, reproducible, fixed evaluation budget
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
import cma

from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Matern, ConstantKernel, WhiteKernel
)
from scipy.optimize import minimize
from scipy.stats import qmc

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

# ---------------- CONFIGURATION ----------------

E5_BUDGET = 200
E5_SEEDS = [42, 123, 2026]
E5_NOISE = [0.0, 0.05]

E5_OUT = Path("experiment_5")
E5_OUT.mkdir(exist_ok=True)

# ---------------- EVALUATOR ----------------

class E5Evaluator:

    def __init__(self, func, sigma, seed):
        self.func = func
        self.sigma = sigma
        self.rng = np.random.default_rng(seed)
        self.X = []
        self.y = []
        self.calls = 0

    def evaluate(self, x):

        if self.calls >= E5_BUDGET:
            raise RuntimeError("Budget exceeded")

        x = np.asarray(x, dtype=float).copy()

        assert x.shape == (CFG.d,)
        assert np.all(np.isfinite(x))
        assert np.all(x >= CFG.low-1e-8)
        assert np.all(x <= CFG.high+1e-8)

        true_y = self.func(x)
        noisy_y = true_y + self.rng.normal(
            0, self.sigma
        )

        self.X.append(x)
        self.y.append(float(noisy_y))
        self.calls += 1

        return float(noisy_y)

# ---------------- INITIALISATION ----------------

def e5_initialise(ev, seed):

    X0 = initial_design(seed)

    for x in X0:
        ev.evaluate(x)

    return np.asarray(ev.X), np.asarray(ev.y)

# ---------------- GP FITTING ----------------

def e5_fit_gp(X, y, sigma, improved=False):

    Xs = (X-CFG.low)/(CFG.high-CFG.low)

    ym = float(np.mean(y))
    ys = max(float(np.std(y)), 1e-8)

    yn = (y-ym)/ys

    if improved:
        kernel = (
            ConstantKernel(
                1.0, (0.01,100.0)
            )
            * Matern(
                length_scale=np.ones(CFG.d)*0.3,
                length_scale_bounds=(0.03,3.0),
                nu=2.5
            )
        )

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=max((sigma/ys)**2,1e-8),
            normalize_y=False,
            n_restarts_optimizer=1,
            random_state=42
        )

    else:
        kernel = Matern(
            length_scale=0.3,
            length_scale_bounds="fixed",
            nu=2.5
        )

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=max((sigma/ys)**2,1e-8),
            normalize_y=False,
            optimizer=None
        )

    gp.fit(Xs,yn)

    return gp,ym,ys

# ---------------- BO ----------------

def e5_run_bo(ev, seed, improved=False):

    rng = np.random.default_rng(seed+7000)
    e5_initialise(ev,seed)

    learned_kernel = None

    while ev.calls < E5_BUDGET:

        X = np.asarray(ev.X)
        y = np.asarray(ev.y)

        # Learn hyperparameters periodically.
        # Between updates, reuse the fitted kernel
        # but still condition on all observations.
        do_tune = (
            improved and (
                learned_kernel is None
                or ev.calls % 25 == 0
            )
        )

        if do_tune:
            gp,ym,ys = e5_fit_gp(
                X,y,ev.sigma,improved=True
            )
            learned_kernel = gp.kernel_

        elif improved:
            Xs = (X-CFG.low)/(CFG.high-CFG.low)
            ym = float(np.mean(y))
            ys = max(float(np.std(y)),1e-8)

            gp = GaussianProcessRegressor(
                kernel=learned_kernel,
                alpha=max((ev.sigma/ys)**2,1e-8),
                normalize_y=False,
                optimizer=None
            )
            gp.fit(Xs,(y-ym)/ys)

        else:
            gp,ym,ys = e5_fit_gp(
                X,y,ev.sigma,improved=False
            )

        # Candidate pool for acquisition search
        n_candidates = 1024 if improved else 512

        candidates = rng.uniform(
            0,1,(n_candidates,CFG.d)
        )

        mu,std = gp.predict(
            candidates,return_std=True
        )

        scores = mu-2.0*std
        z_next = candidates[np.argmin(scores)]

        if improved:
            # Refine acquisition solution locally
            def acquisition(z):
                m,s = gp.predict(
                    np.asarray(z).reshape(1,-1),
                    return_std=True
                )
                return float(m[0]-2.0*s[0])

            result = minimize(
                acquisition,
                z_next,
                method="L-BFGS-B",
                bounds=[(0,1)]*CFG.d,
                options={"maxiter":30}
            )

            if result.success and result.fun < (
                np.min(scores)
            ):
                z_next = result.x

        x_next = (
            CFG.low+z_next*(CFG.high-CFG.low)
        )

        ev.evaluate(x_next)

    X = np.asarray(ev.X)
    y = np.asarray(ev.y)

    observed_idx = int(np.argmin(y))

    # Final posterior recommendation uses the
    # same learned kernel, without extra tuning.
    if improved:
        Xs = (X-CFG.low)/(CFG.high-CFG.low)
        ym = float(np.mean(y))
        ys = max(float(np.std(y)),1e-8)

        gp = GaussianProcessRegressor(
            kernel=learned_kernel,
            alpha=max((ev.sigma/ys)**2,1e-8),
            normalize_y=False,
            optimizer=None
        )
        gp.fit(Xs,(y-ym)/ys)
    else:
        gp,ym,ys = e5_fit_gp(
            X,y,ev.sigma,improved=False
        )

    posterior_means = gp.predict(
        (X-CFG.low)/(CFG.high-CFG.low)
    )

    state_idx = int(np.argmin(posterior_means))

    return {
        "observed_x":X[observed_idx].copy(),
        "state_x":X[state_idx].copy()
    }

# ---------------- CMA-ES ----------------

def e5_run_cma(ev,seed):

    X0,y0 = e5_initialise(ev,seed)

    x0 = X0[np.argmin(y0)].copy()

    es = cma.CMAEvolutionStrategy(
        x0,2.0,
        {
            "seed":seed,
            "bounds":[CFG.low,CFG.high],
            "popsize":8,
            "verbose":-9,
            "verb_disp":0,
            "tolfun":0,
            "tolx":1e-14,
            "tolstagnation":100000
        }
    )

    # Track the best observed point across
    # all generations and any restarts.
    best_x = X0[np.argmin(y0)].copy()
    best_y = float(np.min(y0))

    while ev.calls < E5_BUDGET:

        candidates = es.ask()
        remaining = E5_BUDGET-ev.calls
        batch = candidates[:remaining]

        values = []

        for x in batch:
            val = ev.evaluate(x)
            values.append(val)

            if val < best_y:
                best_y = val
                best_x = np.asarray(x).copy()

        if len(batch)==len(candidates):
            es.tell(candidates,values)
        else:
            break

        if es.stop():
            # Restart at best observed point.
            es = cma.CMAEvolutionStrategy(
                best_x,2.0,
                {
                    "seed":seed+ev.calls+1,
                    "bounds":[CFG.low,CFG.high],
                    "popsize":8,
                    "verbose":-9,
                    "verb_disp":0
                }
            )

    X = np.asarray(ev.X)
    y = np.asarray(ev.y)

    return {
        "observed_x":X[np.argmin(y)].copy(),
        "state_x":best_x.copy()
    }

# ---------------- EXECUTE TRIAL ----------------

def e5_trial(fname,method,noise,seed):

    func = FUNCTIONS[fname]
    sigma = noise*SCALES[fname]

    ev = E5Evaluator(
        func,sigma,seed+100000
    )

    start = time.perf_counter()

    if method=="BO_baseline":
        rec = e5_run_bo(
            ev,seed,improved=False
        )

    elif method=="BO_improved":
        rec = e5_run_bo(
            ev,seed,improved=True
        )

    elif method=="CMA-ES":
        rec = e5_run_cma(ev,seed)

    else:
        raise ValueError(method)

    elapsed = time.perf_counter()-start

    assert ev.calls==E5_BUDGET

    X = np.asarray(ev.X)

    assert np.all(
        (X>=CFG.low-1e-8)
        & (X<=CFG.high+1e-8)
    )

    observed_regret = func(rec["observed_x"])
    state_regret = func(rec["state_x"])
    oracle_regret = min(func(x) for x in X)

    return {
        "function":fname,
        "method":method,
        "noise":noise,
        "seed":seed,
        "budget":ev.calls,
        "observed_regret":observed_regret,
        "state_regret":state_regret,
        "oracle_regret":oracle_regret,
        "runtime":elapsed
    }

# ---------------- RUN EXPERIMENT ----------------

records = []

total = (
    len(FUNCTIONS)
    *len(E5_NOISE)
    *len(E5_SEEDS)
    *3
)

print(f"STARTING EXPERIMENT 5: {total} RUNS")
print("="*70)

for fname in FUNCTIONS:
    for noise in E5_NOISE:
        for seed in E5_SEEDS:
            for method in [
                "BO_baseline",
                "BO_improved",
                "CMA-ES"
            ]:

                result = e5_trial(
                    fname,method,noise,seed
                )

                records.append(result)

                pd.DataFrame(records).to_csv(
                    E5_OUT/"results.csv",
                    index=False
                )

                print(
                    f"{len(records):02d}/{total} "
                    f"{fname:12s} "
                    f"{method:12s} "
                    f"noise={noise:.2f} "
                    f"seed={seed} "
                    f"regret={result['observed_regret']:.5f} "
                    f"state={result['state_regret']:.5f} "
                    f"time={result['runtime']:.2f}s",
                    flush=True
                )

df5 = pd.DataFrame(records)

print("\nEXPERIMENT 5 SUMMARY")
print("="*70)

summary5 = df5.groupby(
    ["function","noise","method"]
)[["observed_regret","state_regret","runtime"]].agg(
    ["mean","std"]
).round(5)

print(summary5.to_string())

print("\nBUDGET CHECK")
print(df5.groupby("method")["budget"].agg(
    ["min","max","count"]
))

print("\nEXPERIMENT 5 COMPLETE")
print("Saved:",E5_OUT/"results.csv")


In [ ]:

# ============================================================
# EXPERIMENT 6A
# Fixed-budget exploration vs re-evaluation
# Improved BO vs CMA-ES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, json, platform
from pathlib import Path

import cma
import sklearn
import scipy

from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

# ---------------- CONFIGURATION ----------------

E6_BUDGET = 400
E6_FUNCTIONS = ["Sphere", "Rosenbrock"]
E6_SEEDS = [42, 123]
E6_NOISE = [0.05]
E6_FRACTIONS = [0.0, 0.10, 0.25]

E6_K = 5
E6_OUT = Path("experiment_6")
E6_OUT.mkdir(exist_ok=True)

assert CFG.d == 5
assert CFG.low == -5.0
assert CFG.high == 5.0
assert CFG.n_initial == 10

# ---------------- AUDITED EVALUATOR ----------------

class E6Evaluator:

    def __init__(self, func, sigma, seed, budget):
        self._func = func
        self.sigma = float(sigma)
        self.budget = int(budget)
        self.rng = np.random.default_rng(seed)

        self.X = []
        self.y = []
        self.phase = []
        self.calls = 0

    def evaluate(self, x, phase="search"):

        if self.calls >= self.budget:
            raise RuntimeError("Budget exceeded")

        x = np.asarray(x, dtype=float).copy()

        assert x.shape == (CFG.d,)
        assert np.all(np.isfinite(x))
        assert np.all(x >= CFG.low - 1e-8)
        assert np.all(x <= CFG.high + 1e-8)
        assert phase in ["search", "validation"]

        # True objective is used only inside evaluator.
        true_y = self._func(x)

        observed_y = (
            true_y + self.rng.normal(0, self.sigma)
        )

        self.X.append(x)
        self.y.append(float(observed_y))
        self.phase.append(phase)
        self.calls += 1

        return float(observed_y)

# ---------------- COMMON INITIAL DESIGN ----------------

def e6_initialise(ev, seed):
    X0 = initial_design(seed)

    y0 = [
        ev.evaluate(x, "search")
        for x in X0
    ]

    return np.asarray(X0), np.asarray(y0)

# ---------------- IMPROVED BO ----------------

def e6_gp(X, y, sigma, kernel=None, tune=False):

    Xs = (X - CFG.low) / (CFG.high - CFG.low)

    ym = float(np.mean(y))
    ys = max(float(np.std(y)), 1e-8)

    yn = (y - ym) / ys

    if kernel is None:
        kernel = (
            ConstantKernel(1.0, (0.01, 100.0))
            * Matern(
                length_scale=np.ones(CFG.d) * 0.3,
                length_scale_bounds=(0.03, 3.0),
                nu=2.5
            )
        )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=max((sigma / ys)**2, 1e-8),
        normalize_y=False,
        optimizer="fmin_l_bfgs_b" if tune else None,
        n_restarts_optimizer=1 if tune else 0,
        random_state=42
    )

    gp.fit(Xs, yn)

    return gp

def e6_run_bo(ev, seed, search_budget):

    rng = np.random.default_rng(seed + 7000)

    e6_initialise(ev, seed)

    learned_kernel = None

    while ev.calls < search_budget:

        X = np.asarray(ev.X)
        y = np.asarray(ev.y)

        tune = (
            learned_kernel is None
            or ev.calls % 25 == 0
        )

        gp = e6_gp(
            X, y, ev.sigma,
            kernel=None if tune else learned_kernel,
            tune=tune
        )

        if tune:
            learned_kernel = gp.kernel_

        candidates = rng.uniform(
            0, 1, (1024, CFG.d)
        )

        mu, std = gp.predict(
            candidates, return_std=True
        )

        scores = mu - 2.0 * std

        z_next = candidates[np.argmin(scores)]

        # Local acquisition refinement
        def acquisition(z):
            m, s = gp.predict(
                np.asarray(z).reshape(1, -1),
                return_std=True
            )
            return float(m[0] - 2.0*s[0])

        result = minimize(
            acquisition,
            z_next,
            method="L-BFGS-B",
            bounds=[(0, 1)] * CFG.d,
            options={"maxiter": 30}
        )

        if (
            result.success
            and result.fun < np.min(scores)
        ):
            z_next = result.x

        x_next = (
            CFG.low +
            z_next * (CFG.high - CFG.low)
        )

        ev.evaluate(x_next, "search")

    assert ev.calls == search_budget

# ---------------- CMA-ES ----------------

def e6_run_cma(ev, seed, search_budget):

    X0, y0 = e6_initialise(ev, seed)

    best_x = X0[np.argmin(y0)].copy()
    best_y = float(np.min(y0))

    def make_es(center, es_seed):
        return cma.CMAEvolutionStrategy(
            np.clip(center, CFG.low, CFG.high),
            2.0,
            {
                "seed": int(es_seed),
                "bounds": [CFG.low, CFG.high],
                "popsize": 8,
                "verbose": -9,
                "verb_disp": 0,
                "tolfun": 0,
                "tolx": 1e-14,
                "tolstagnation": 100000
            }
        )

    es = make_es(best_x, seed)

    while ev.calls < search_budget:

        candidates = es.ask()
        remaining = search_budget - ev.calls
        batch = candidates[:remaining]

        values = []

        for x in batch:
            value = ev.evaluate(x, "search")
            values.append(value)

            if value < best_y:
                best_y = value
                best_x = np.asarray(x).copy()

        if len(batch) == len(candidates):
            es.tell(candidates, values)
        else:
            # Final incomplete generation is counted.
            break

        if es.stop():
            es = make_es(
                best_x,
                seed + ev.calls + 1
            )

    assert ev.calls == search_budget

# ---------------- VALIDATION POLICY ----------------

def e6_validate(ev, search_budget, validation_budget):

    # Only observations collected during search
    # are eligible for the initial shortlist.
    X = np.asarray(ev.X[:search_budget])
    y = np.asarray(ev.y[:search_budget])

    shortlist = np.argsort(y)[:E6_K]

    if validation_budget == 0:
        selected_idx = int(shortlist[0])
        return {
            "selected_x": X[selected_idx].copy(),
            "selected_idx": selected_idx,
            "shortlist": shortlist.tolist()
        }

    # Initial observations count towards averages.
    sums = y[shortlist].copy()
    counts = np.ones(E6_K, dtype=int)

    for j in range(validation_budget):

        slot = j % E6_K
        idx = shortlist[slot]

        new_y = ev.evaluate(
            X[idx],
            phase="validation"
        )

        sums[slot] += new_y
        counts[slot] += 1

    averages = sums / counts

    winning_slot = int(np.argmin(averages))
    selected_idx = int(shortlist[winning_slot])

    return {
        "selected_x": X[selected_idx].copy(),
        "selected_idx": selected_idx,
        "shortlist": shortlist.tolist()
    }

# ---------------- COMPLETE TRIAL ----------------

def e6_trial(fname, method, noise, seed, fraction):

    func = FUNCTIONS[fname]
    sigma = noise * SCALES[fname]

    validation_budget = int(
        round(E6_BUDGET * fraction)
    )
    search_budget = E6_BUDGET - validation_budget

    assert search_budget >= CFG.n_initial
    assert validation_budget == 0 or (
        validation_budget >= E6_K
    )

    # Separate random streams for search and validation.
    # This makes the search prefix comparable across
    # budget policies with the same method and seed.
    search_ev = E6Evaluator(
        func, sigma, seed + 100000, search_budget
    )

    start = time.perf_counter()

    if method == "BO":
        e6_run_bo(search_ev, seed, search_budget)
    elif method == "CMA-ES":
        e6_run_cma(search_ev, seed, search_budget)
    else:
        raise ValueError(method)

    search_time = time.perf_counter() - start

    # Recreate a full-budget evaluator for validation,
    # copying search observations without re-evaluating.
    # This preserves the original charged search calls.
    ev = E6Evaluator(
        func, sigma, seed + 200000, E6_BUDGET
    )

    ev.X = [x.copy() for x in search_ev.X]
    ev.y = list(search_ev.y)
    ev.phase = list(search_ev.phase)
    ev.calls = search_ev.calls

    start_validation = time.perf_counter()

    recommendation = e6_validate(
        ev, search_budget, validation_budget
    )

    validation_time = (
        time.perf_counter() - start_validation
    )

    # ---------------- AUDIT ----------------

    assert ev.calls == E6_BUDGET
    assert len(ev.X) == E6_BUDGET
    assert len(ev.y) == E6_BUDGET

    assert ev.phase.count("search") == search_budget
    assert ev.phase.count("validation") == validation_budget

    selected_x = recommendation["selected_x"]

    # Independent scoring, after optimisation.
    selected_regret = func(selected_x)

    search_X = np.asarray(
        ev.X[:search_budget]
    )

    # Oracle diagnostic, not used for selection.
    oracle_regret = min(
        func(x) for x in search_X
    )

    selection_gap = (
        selected_regret - oracle_regret
    )

    assert selection_gap >= -1e-7

    record = {
        "function": fname,
        "method": method,
        "noise": noise,
        "sigma": sigma,
        "seed": seed,
        "fraction": fraction,
        "search_budget": search_budget,
        "validation_budget": validation_budget,
        "total_evaluations": ev.calls,
        "selected_regret": selected_regret,
        "oracle_regret": oracle_regret,
        "selection_gap": selection_gap,
        "search_time": search_time,
        "validation_time": validation_time,
        "total_time": search_time + validation_time,
        "selected_idx": recommendation["selected_idx"]
    }

    trace = pd.DataFrame({
        "function": fname,
        "method": method,
        "noise": noise,
        "seed": seed,
        "fraction": fraction,
        "evaluation": np.arange(1, E6_BUDGET + 1),
        "phase": ev.phase,
        "observed_y": ev.y,
        **{
            f"x{j}": np.asarray(ev.X)[:, j]
            for j in range(CFG.d)
        }
    })

    return record, trace

print("EXPERIMENT 6 FRAMEWORK DEFINED")


In [ ]:

# ============================================================
# EXPERIMENT 6A: RUN AND ANALYSE
# ============================================================

records6 = []
traces6 = []

total = (
    len(E6_FUNCTIONS)
    * len(E6_NOISE)
    * len(E6_SEEDS)
    * 2
    * len(E6_FRACTIONS)
)

print(f"STARTING EXPERIMENT 6A: {total} RUNS")
print("=" * 75)

for fname in E6_FUNCTIONS:
    for noise in E6_NOISE:
        for seed in E6_SEEDS:
            for method in ["BO", "CMA-ES"]:
                for fraction in E6_FRACTIONS:

                    result, trace = e6_trial(
                        fname, method, noise,
                        seed, fraction
                    )

                    records6.append(result)
                    traces6.append(trace)

                    pd.DataFrame(records6).to_csv(
                        E6_OUT / "results.csv",
                        index=False
                    )

                    print(
                        f"{len(records6):02d}/{total} "
                        f"{fname:12s} "
                        f"{method:6s} "
                        f"seed={seed} "
                        f"validation={fraction:.0%} "
                        f"regret={result['selected_regret']:.5f} "
                        f"gap={result['selection_gap']:.5f} "
                        f"time={result['total_time']:.2f}s",
                        flush=True
                    )

df6 = pd.DataFrame(records6)

trace6 = pd.concat(
    traces6, ignore_index=True
)

trace6.to_csv(
    E6_OUT / "observations.csv",
    index=False
)

# ---------------- AUDIT ----------------

assert len(df6) == total
assert (df6["total_evaluations"] == E6_BUDGET).all()

assert (
    df6["search_budget"]
    + df6["validation_budget"]
    == E6_BUDGET
).all()

assert (df6["selection_gap"] >= -1e-7).all()

print("\nALL BUDGET CHECKS PASSED")

# ---------------- SUMMARY ----------------

summary6 = df6.groupby(
    ["function", "method", "fraction"]
).agg(
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    mean_gap=("selection_gap", "mean"),
    mean_oracle=("oracle_regret", "mean"),
    mean_runtime=("total_time", "mean")
).reset_index()

print("\nEXPERIMENT 6A SUMMARY")
print("=" * 75)
print(summary6.round(5).to_string(index=False))

# ---------------- PLOT 1: TRUE REGRET ----------------

fig, axes = plt.subplots(
    1, len(E6_FUNCTIONS),
    figsize=(12, 4.5)
)

for ax, fname in zip(
    np.atleast_1d(axes), E6_FUNCTIONS
):
    for method in ["BO", "CMA-ES"]:
        subset = summary6[
            (summary6["function"] == fname)
            & (summary6["method"] == method)
        ]

        ax.plot(
            subset["fraction"] * 100,
            subset["mean_regret"],
            marker="o",
            label=method
        )

    ax.set_title(fname)
    ax.set_xlabel("Validation budget (%)")
    ax.set_ylabel("Mean true simple regret")
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle(
    "Experiment 6A: Final Recommendation Quality"
)
plt.tight_layout()
plt.savefig(
    E6_OUT / "true_regret.png",
    dpi=160
)
plt.show()

# ---------------- PLOT 2: SELECTION GAP ----------------

fig, axes = plt.subplots(
    1, len(E6_FUNCTIONS),
    figsize=(12, 4.5)
)

for ax, fname in zip(
    np.atleast_1d(axes), E6_FUNCTIONS
):
    for method in ["BO", "CMA-ES"]:
        subset = summary6[
            (summary6["function"] == fname)
            & (summary6["method"] == method)
        ]

        ax.plot(
            subset["fraction"] * 100,
            subset["mean_gap"],
            marker="o",
            label=method
        )

    ax.set_title(fname)
    ax.set_xlabel("Validation budget (%)")
    ax.set_ylabel("Mean selection gap")
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle(
    "Experiment 6A: Selection Error"
)
plt.tight_layout()
plt.savefig(
    E6_OUT / "selection_gap.png",
    dpi=160
)
plt.show()

# ---------------- SAVE METADATA ----------------

metadata6 = {
    "experiment": "6A",
    "purpose": "Fixed-budget replication pilot",
    "budget": E6_BUDGET,
    "functions": E6_FUNCTIONS,
    "seeds": E6_SEEDS,
    "noise_fractions": E6_NOISE,
    "validation_fractions": E6_FRACTIONS,
    "shortlist_size": E6_K,
    "dimension": CFG.d,
    "noise_scales": {
        k: SCALES[k] for k in E6_FUNCTIONS
    },
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "cma": cma.__version__
}

with open(
    E6_OUT / "metadata.json", "w"
) as f:
    json.dump(metadata6, f, indent=2)

print("\nEXPERIMENT 6A COMPLETE")
print("Files saved in:", E6_OUT)


In [ ]:

# ============================================================
# EXPERIMENT 6A AUDIT
# Budget, prefixes, noise, selection, reproducibility
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

AUDIT_OUT = Path("experiment_6/audit")
AUDIT_OUT.mkdir(parents=True, exist_ok=True)

print("EXPERIMENT 6A AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD ORIGINAL RESULTS
# ------------------------------------------------------------

results = pd.read_csv("experiment_6/results.csv")
traces = pd.read_csv("experiment_6/observations.csv")

assert len(results) == 24
assert len(traces) == 24 * E6_BUDGET

print("PASS: Expected number of trials and observations")

# ------------------------------------------------------------
# 2. BUDGET ACCOUNTING
# ------------------------------------------------------------

assert (results["total_evaluations"] == E6_BUDGET).all()

assert (
    results["search_budget"] +
    results["validation_budget"]
    == E6_BUDGET
).all()

for key, group in traces.groupby(
    ["function", "method", "noise", "seed", "fraction"]
):
    row = results[
        (results["function"] == key[0])
        & (results["method"] == key[1])
        & np.isclose(results["noise"], key[2])
        & (results["seed"] == key[3])
        & np.isclose(results["fraction"], key[4])
    ]

    assert len(row) == 1
    assert len(group) == E6_BUDGET

    assert (
        (group["phase"] == "search").sum()
        == int(row.iloc[0]["search_budget"])
    )

    assert (
        (group["phase"] == "validation").sum()
        == int(row.iloc[0]["validation_budget"])
    )

print("PASS: All search and validation budgets")

# ------------------------------------------------------------
# 3. SEARCH-PREFIX CONSISTENCY
# ------------------------------------------------------------

prefix_checks = []

for fname in E6_FUNCTIONS:
    for method in ["BO", "CMA-ES"]:
        for seed in E6_SEEDS:

            subset = traces[
                (traces["function"] == fname)
                & (traces["method"] == method)
                & (traces["seed"] == seed)
            ]

            base = subset[
                np.isclose(subset["fraction"], 0.0)
            ].sort_values("evaluation")

            for fraction in [0.10, 0.25]:

                other = subset[
                    np.isclose(
                        subset["fraction"], fraction
                    )
                ].sort_values("evaluation")

                search_count = int(
                    E6_BUDGET*(1-fraction)
                )

                cols = (
                    [f"x{i}" for i in range(CFG.d)]
                    + ["observed_y"]
                )

                a = base[cols].iloc[:search_count].to_numpy()
                b = other[cols].iloc[:search_count].to_numpy()

                same = np.allclose(
                    a, b,
                    rtol=0, atol=1e-9
                )

                prefix_checks.append({
                    "function": fname,
                    "method": method,
                    "seed": seed,
                    "fraction": fraction,
                    "identical_prefix": same
                })

prefix_df = pd.DataFrame(prefix_checks)

print("\nSEARCH PREFIX CHECK")
print(prefix_df.to_string(index=False))

prefix_df.to_csv(
    AUDIT_OUT / "prefix_checks.csv",
    index=False
)

assert prefix_df["identical_prefix"].all()
print("PASS: All search prefixes identical")

# ------------------------------------------------------------
# 4. VALIDATION REPLICATION CHECK
# ------------------------------------------------------------

for key, group in traces.groupby(
    ["function", "method", "noise", "seed", "fraction"]
):

    validation = group[
        group["phase"] == "validation"
    ]

    if validation.empty:
        continue

    search = group[
        group["phase"] == "search"
    ]

    search_X = search[
        [f"x{i}" for i in range(CFG.d)]
    ].to_numpy()

    validation_X = validation[
        [f"x{i}" for i in range(CFG.d)]
    ].to_numpy()

    for x in validation_X:

        distances = np.linalg.norm(
            search_X-x, axis=1
        )

        assert np.min(distances) < 1e-9

print("PASS: Validation only revisits searched candidates")

# ------------------------------------------------------------
# 5. RECOMPUTE RECOMMENDATIONS FROM RAW DATA
# ------------------------------------------------------------

selection_records = []

for key, group in traces.groupby(
    ["function", "method", "noise", "seed", "fraction"]
):

    fname, method, noise, seed, fraction = key

    group = group.sort_values("evaluation")

    search = group[
        group["phase"] == "search"
    ]

    validation = group[
        group["phase"] == "validation"
    ]

    X = search[
        [f"x{i}" for i in range(CFG.d)]
    ].to_numpy()

    y = search["observed_y"].to_numpy()

    shortlist = np.argsort(y)[:E6_K]

    if len(validation) == 0:
        selected_idx = int(shortlist[0])
    else:
        sums = y[shortlist].copy()
        counts = np.ones(E6_K, dtype=int)

        validation_y = validation[
            "observed_y"
        ].to_numpy()

        for j, value in enumerate(validation_y):
            slot = j % E6_K
            sums[slot] += value
            counts[slot] += 1

        selected_idx = int(
            shortlist[np.argmin(sums/counts)]
        )

    selected_x = X[selected_idx]
    recomputed_regret = FUNCTIONS[fname](selected_x)

    original = results[
        (results["function"] == fname)
        & (results["method"] == method)
        & (np.isclose(results["noise"], noise))
        & (results["seed"] == seed)
        & (np.isclose(results["fraction"], fraction))
    ].iloc[0]

    assert np.isclose(
        recomputed_regret,
        original["selected_regret"],
        rtol=0, atol=1e-7
    )

    selection_records.append({
        "function": fname,
        "method": method,
        "seed": seed,
        "fraction": fraction,
        "selected_idx": selected_idx,
        "selected_regret": recomputed_regret
    })

selection_df = pd.DataFrame(selection_records)

selection_df.to_csv(
    AUDIT_OUT / "selection_checks.csv",
    index=False
)

print("PASS: All recommendations reconstructed correctly")

# ------------------------------------------------------------
# 6. INDEPENDENT NOISE CHECK
# ------------------------------------------------------------

# Check that repeated observations at the same x
# are not simply identical copies.

repeated_groups = 0
different_measurements = 0

for key, group in traces.groupby(
    ["function", "method", "noise", "seed", "fraction"]
):

    if key[2] == 0:
        continue

    coordinate_cols = [
        f"x{i}" for i in range(CFG.d)
    ]

    for _, same_x in group.groupby(
        coordinate_cols
    ):

        if len(same_x) > 1:
            repeated_groups += 1

            if same_x["observed_y"].nunique() > 1:
                different_measurements += 1

print(
    f"Repeated candidate groups: {repeated_groups}"
)
print(
    f"Groups with varying observations: "
    f"{different_measurements}"
)

assert different_measurements > 0

print("PASS: Repeated measurements contain fresh noise")

# ------------------------------------------------------------
# 7. REPRODUCIBILITY TEST
# ------------------------------------------------------------

print("\nREPRODUCIBILITY TEST")

test_cases = [
    ("Sphere", "CMA-ES", 0.05, 42, 0.10),
    ("Rosenbrock", "BO", 0.05, 123, 0.10)
]

for case in test_cases:

    fname, method, noise, seed, fraction = case

    new_result, new_trace = e6_trial(
        fname, method, noise, seed, fraction
    )

    old = results[
        (results["function"] == fname)
        & (results["method"] == method)
        & (results["seed"] == seed)
        & np.isclose(results["fraction"], fraction)
    ].iloc[0]

    assert np.isclose(
        new_result["selected_regret"],
        old["selected_regret"],
        rtol=0, atol=1e-7
    )

    print(
        f"PASS: {fname}, {method}, "
        f"seed={seed}, fraction={fraction}"
    )

# ------------------------------------------------------------
# 8. SAVE AUDIT SUMMARY
# ------------------------------------------------------------

audit_summary = {
    "expected_trials": 24,
    "observed_trials": len(results),
    "budget_verified": True,
    "prefixes_verified": True,
    "recommendations_verified": True,
    "fresh_noise_verified": True,
    "reproducibility_verified": True
}

import json

with open(
    AUDIT_OUT / "audit_summary.json", "w"
) as f:
    json.dump(audit_summary, f, indent=2)

print("\n" + "=" * 70)
print("EXPERIMENT 6A AUDIT PASSED")
print("=" * 70)
print("Saved audit files in:", AUDIT_OUT)


In [ ]:

# ============================================================
# EXPERIMENT 6A: RECOMMENDATION DIAGNOSTIC
# ============================================================

selection = pd.read_csv(
    "experiment_6/audit/selection_checks.csv"
)

print("FINAL SELECTED CANDIDATES")
print("=" * 65)

print(
    selection.sort_values(
        ["function", "method", "seed", "fraction"]
    ).to_string(index=False)
)

print("\nSAME CANDIDATE ACROSS POLICIES?")
print("=" * 65)

for (fname, method, seed), group in selection.groupby(
    ["function", "method", "seed"]
):

    group = group.sort_values("fraction")

    indices = group["selected_idx"].tolist()

    print(
        f"{fname:12s} | {method:6s} | "
        f"seed={seed} | "
        f"selected indices={indices} | "
        f"all_same={len(set(indices)) == 1}"
    )


In [ ]:

# ============================================================
# EXPERIMENT 6B-1
# Reusable search trajectories
# ============================================================

import numpy as np
import pandas as pd
import time
import json
import platform
from pathlib import Path

OUT6B = Path("experiment_6b")
SEARCH_DIR = OUT6B / "search"
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

BUDGETS6B = [400, 600]
MAX_SEARCH6B = max(BUDGETS6B)
FRACTIONS6B = [0.0, 0.10, 0.25]
SEEDS6B = [42, 123]
NOISE6B = [0.05]
FUNCTIONS6B = list(FUNCTIONS.keys())
METHODS6B = ["BO", "CMA-ES"]
K6B = 5

def search_path(fname, method, noise, seed):
    return SEARCH_DIR / (
        f"{fname}_{method}_n{noise:.3f}"
        f"_seed{seed}_B{MAX_SEARCH6B}.npz"
    )

def generate_search(fname, method, noise, seed):

    func = FUNCTIONS[fname]
    sigma = noise * SCALES[fname]

    ev = E6Evaluator(
        func,
        sigma,
        seed + 100000,
        MAX_SEARCH6B
    )

    start = time.perf_counter()

    if method == "BO":
        e6_run_bo(ev, seed, MAX_SEARCH6B)
    else:
        e6_run_cma(ev, seed, MAX_SEARCH6B)

    elapsed = time.perf_counter() - start

    assert ev.calls == MAX_SEARCH6B
    assert len(ev.X) == MAX_SEARCH6B

    X = np.asarray(ev.X)
    y = np.asarray(ev.y)

    assert np.isfinite(X).all()
    assert np.isfinite(y).all()
    assert np.all(X >= CFG.low - 1e-8)
    assert np.all(X <= CFG.high + 1e-8)

    return X, y, elapsed


search_times = []

total_searches = (
    len(FUNCTIONS6B)
    * len(METHODS6B)
    * len(NOISE6B)
    * len(SEEDS6B)
)

print("EXPERIMENT 6B: SEARCH GENERATION")
print("Maximum budget:", MAX_SEARCH6B)
print("Search trajectories:", total_searches)
print("=" * 70)

counter = 0

for fname in FUNCTIONS6B:
    for noise in NOISE6B:
        for seed in SEEDS6B:
            for method in METHODS6B:

                counter += 1
                path = search_path(
                    fname, method, noise, seed
                )

                if path.exists():
                    with np.load(path) as saved:
                        X = saved["X"]
                        y = saved["y"]
                        elapsed = float(saved["elapsed"])

                    assert len(y) == MAX_SEARCH6B
                    status = "LOADED"

                else:
                    X, y, elapsed = generate_search(
                        fname, method, noise, seed
                    )

                    np.savez_compressed(
                        path,
                        X=X,
                        y=y,
                        elapsed=elapsed
                    )
                    status = "NEW"

                search_times.append({
                    "function": fname,
                    "method": method,
                    "noise": noise,
                    "seed": seed,
                    "max_search_budget": MAX_SEARCH6B,
                    "runtime_seconds": elapsed
                })

                print(
                    f"{counter:02d}/{total_searches} "
                    f"{fname:12s} "
                    f"{method:6s} "
                    f"seed={seed} "
                    f"time={elapsed:.2f}s "
                    f"[{status}]",
                    flush=True
                )

pd.DataFrame(search_times).to_csv(
    OUT6B / "search_times.csv",
    index=False
)

print("\nALL SEARCH TRAJECTORIES SAVED")


In [ ]:

# ============================================================
# EXPERIMENT 6B: VALIDATION POLICIES
# ============================================================

records6b = []
validation_traces = []

for fname in FUNCTIONS6B:
    func = FUNCTIONS[fname]

    for noise in NOISE6B:
        sigma = noise * SCALES[fname]

        for seed in SEEDS6B:
            for method in METHODS6B:

                path = search_path(
                    fname, method, noise, seed
                )

                with np.load(path) as saved:
                    X_full = saved["X"]
                    y_full = saved["y"]
                    search_runtime = float(
                        saved["elapsed"]
                    )

                assert len(y_full) == MAX_SEARCH6B

                for budget in BUDGETS6B:
                    for fraction in FRACTIONS6B:

                        n_val = int(
                            round(budget * fraction)
                        )
                        n_search = budget - n_val

                        assert n_search >= CFG.n_initial
                        assert n_search <= len(y_full)

                        X = X_full[:n_search]
                        y = y_full[:n_search]

                        shortlist = np.argsort(
                            y, kind="stable"
                        )[:K6B]

                        sums = y[shortlist].copy()
                        counts = np.ones(K6B, dtype=int)

                        # Independent validation stream,
                        # shared across fractions for
                        # the same experimental condition.
                        vrng = np.random.default_rng(
                            np.random.SeedSequence([
                                6006,
                                seed,
                                int(noise * 100000),
                                budget,
                                FUNCTIONS6B.index(fname),
                                METHODS6B.index(method)
                            ])
                        )

                        val_start = time.perf_counter()

                        for j in range(n_val):
                            slot = j % K6B
                            idx = int(shortlist[slot])

                            # True objective is accessed
                            # only by the simulated evaluator,
                            # never by candidate selection.
                            observed = (
                                func(X[idx])
                                + vrng.normal(0, sigma)
                            )

                            sums[slot] += observed
                            counts[slot] += 1

                            validation_traces.append({
                                "function": fname,
                                "method": method,
                                "noise": noise,
                                "seed": seed,
                                "budget": budget,
                                "fraction": fraction,
                                "validation_call": j + 1,
                                "candidate_idx": idx,
                                "observed_y": observed
                            })

                        means = sums / counts

                        winning_slot = int(
                            np.argmin(means)
                        )
                        selected_idx = int(
                            shortlist[winning_slot]
                        )

                        validation_runtime = (
                            time.perf_counter()
                            - val_start
                        )

                        # Independent offline scoring
                        selected_regret = func(
                            X[selected_idx]
                        )

                        oracle_regret = min(
                            func(x) for x in X
                        )

                        gap = (
                            selected_regret - oracle_regret
                        )

                        assert gap >= -1e-7
                        assert n_search + n_val == budget
                        assert selected_idx < n_search

                        records6b.append({
                            "function": fname,
                            "method": method,
                            "noise": noise,
                            "sigma": sigma,
                            "seed": seed,
                            "budget": budget,
                            "fraction": fraction,
                            "search_evals": n_search,
                            "validation_evals": n_val,
                            "total_evals": budget,
                            "selected_idx": selected_idx,
                            "selected_regret":
                                selected_regret,
                            "oracle_regret":
                                oracle_regret,
                            "selection_gap": gap,
                            "validation_runtime":
                                validation_runtime,
                            "search_full_runtime":
                                search_runtime
                        })

df6b = pd.DataFrame(records6b)

df6b.to_csv(
    OUT6B / "results.csv",
    index=False
)

pd.DataFrame(validation_traces).to_csv(
    OUT6B / "validation_observations.csv",
    index=False
)

expected = (
    len(FUNCTIONS6B)
    * len(METHODS6B)
    * len(NOISE6B)
    * len(SEEDS6B)
    * len(BUDGETS6B)
    * len(FRACTIONS6B)
)

assert len(df6b) == expected
assert (
    df6b["search_evals"]
    + df6b["validation_evals"]
    == df6b["total_evals"]
).all()

print("ALL BUDGET CHECKS PASSED")
print("Policy-level results:", len(df6b))

summary6b = df6b.groupby(
    ["function", "budget", "method", "fraction"]
).agg(
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    mean_gap=("selection_gap", "mean"),
    mean_oracle=("oracle_regret", "mean")
).reset_index()

print("\nEXPERIMENT 6B SUMMARY")
print("=" * 80)
print(
    summary6b.round(5).to_string(index=False)
)

summary6b.to_csv(
    OUT6B / "summary.csv",
    index=False
)

metadata6b = {
    "experiment": "6B-1",
    "budgets": BUDGETS6B,
    "max_search": MAX_SEARCH6B,
    "fractions": FRACTIONS6B,
    "seeds": SEEDS6B,
    "functions": FUNCTIONS6B,
    "methods": METHODS6B,
    "noise_levels": NOISE6B,
    "shortlist_size": K6B,
    "dimension": CFG.d,
    "noise_scales": SCALES,
    "search_policy": "Experiment 6A BO/CMA-ES",
    "validation_policy": "Top-5 noisy observations, round-robin replication",
    "runtime_note": (
        "Full trajectory runtime is not the runtime "
        "of each shorter budget policy."
    ),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "cma": cma.__version__
}

with open(
    OUT6B / "metadata.json", "w"
) as f:
    json.dump(metadata6b, f, indent=2)

print("\nEXPERIMENT 6B-1 COMPLETE")


In [ ]:

# ============================================================
# EXPERIMENT 6B: PLOTS
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    2, 2, figsize=(13, 9)
)

for ax, fname in zip(
    axes.flat, FUNCTIONS6B
):

    for method in METHODS6B:
        for budget in BUDGETS6B:

            subset = summary6b[
                (summary6b["function"] == fname)
                & (summary6b["method"] == method)
                & (summary6b["budget"] == budget)
            ].sort_values("fraction")

            ax.plot(
                subset["fraction"] * 100,
                subset["mean_regret"],
                marker="o",
                label=f"{method}, B={budget}"
            )

    ax.set_title(fname)
    ax.set_xlabel("Validation allocation (%)")
    ax.set_ylabel("Mean final true regret")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle(
    "Experiment 6B: Budget and Validation Sensitivity"
)
plt.tight_layout()

plt.savefig(
    OUT6B / "budget_validation.png",
    dpi=160
)
plt.show()

print("Plot saved.")


In [ ]:

# ============================================================
# EXPERIMENT 6C
# Shortlist quality and selection-error decomposition
# Uses saved Experiment 6B results only
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUT6C = Path("experiment_6c")
OUT6C.mkdir(exist_ok=True)

# Load Experiment 6B outputs
results = pd.read_csv("experiment_6b/results.csv")
validation = pd.read_csv(
    "experiment_6b/validation_observations.csv"
)

diagnostics = []

print("EXPERIMENT 6C: SHORTLIST DIAGNOSTICS")
print("=" * 75)

for _, row in results.iterrows():

    fname = row["function"]
    method = row["method"]
    noise = float(row["noise"])
    seed = int(row["seed"])
    budget = int(row["budget"])
    fraction = float(row["fraction"])
    n_search = int(row["search_evals"])

    # Load original saved search trajectory
    path = (
        Path("experiment_6b/search")
        / (
            f"{fname}_{method}_n{noise:.3f}"
            f"_seed{seed}_B{MAX_SEARCH6B}.npz"
        )
    )

    with np.load(path) as saved:
        X = saved["X"][:n_search]
        y = saved["y"][:n_search]

    func = FUNCTIONS[fname]

    # Offline oracle scoring:
    # true values are never passed to the optimiser.
    true_values = np.array([
        func(x) for x in X
    ])

    oracle_idx = int(np.argmin(true_values))
    oracle_regret = true_values[oracle_idx]

    # Reconstruct the original noisy shortlist
    shortlist = np.argsort(
        y, kind="stable"
    )[:K6B]

    shortlist_true = true_values[shortlist]

    best_shortlist_regret = float(
        np.min(shortlist_true)
    )

    shortlist_loss = (
        best_shortlist_regret-oracle_regret
    )

    shortlist_coverage = (
        oracle_idx in shortlist
    )

    # Selected candidate from Experiment 6B
    selected_idx = int(row["selected_idx"])
    selected_regret = true_values[selected_idx]

    ranking_loss = (
        selected_regret-best_shortlist_regret
    )

    total_gap = (
        selected_regret-oracle_regret
    )

    # Validate decomposition
    assert np.isclose(
        shortlist_loss+ranking_loss,
        total_gap,
        rtol=0, atol=1e-7
    )

    assert shortlist_loss >= -1e-7
    assert ranking_loss >= -1e-7

    # Noisy best candidate before validation
    noisy_best_idx = int(np.argmin(y))
    noisy_best_regret = true_values[noisy_best_idx]

    diagnostics.append({
        "function":fname,
        "method":method,
        "noise":noise,
        "seed":seed,
        "budget":budget,
        "fraction":fraction,
        "search_evals":n_search,
        "shortlist_coverage":int(shortlist_coverage),
        "oracle_regret":oracle_regret,
        "best_shortlist_regret":best_shortlist_regret,
        "selected_regret":selected_regret,
        "noisy_best_regret":noisy_best_regret,
        "shortlist_loss":shortlist_loss,
        "ranking_loss":ranking_loss,
        "total_gap":total_gap,
        "shortlist_true_range":
            float(np.ptp(shortlist_true)),
        "shortlist_true_std":
            float(np.std(shortlist_true))
    })

df6c = pd.DataFrame(diagnostics)

df6c.to_csv(
    OUT6C/"diagnostics.csv",
    index=False
)

print("\nSHORTLIST COVERAGE AND ERROR DECOMPOSITION")
print("=" * 75)

summary6c = df6c.groupby(
    ["function","budget","method","fraction"]
).agg(
    coverage_rate=("shortlist_coverage","mean"),
    mean_shortlist_loss=("shortlist_loss","mean"),
    mean_ranking_loss=("ranking_loss","mean"),
    mean_total_gap=("total_gap","mean"),
    mean_selected_regret=("selected_regret","mean")
).reset_index()

print(
    summary6c.round(5).to_string(index=False)
)

summary6c.to_csv(
    OUT6C/"summary.csv",
    index=False
)

print("\nEXPERIMENT 6C DIAGNOSTICS COMPLETE")
print("Rows analysed:",len(df6c))
print("Saved:",OUT6C/"diagnostics.csv")


In [ ]:

# ============================================================
# EXPERIMENT 6C: ERROR DECOMPOSITION PLOTS
# ============================================================

fig, axes = plt.subplots(
    2, 2, figsize=(13, 9)
)

for ax, fname in zip(
    axes.flat, FUNCTIONS6B
):

    subset = summary6c[
        (summary6c["function"]==fname)
        & (summary6c["budget"]==600)
    ]

    labels = []
    shortlist_losses = []
    ranking_losses = []

    for method in ["BO","CMA-ES"]:
        for fraction in [0.0,0.10,0.25]:

            row = subset[
                (subset["method"]==method)
                & np.isclose(
                    subset["fraction"],fraction
                )
            ].iloc[0]

            labels.append(
                f"{method}\n{fraction:.0%}"
            )

            shortlist_losses.append(
                row["mean_shortlist_loss"]
            )

            ranking_losses.append(
                row["mean_ranking_loss"]
            )

    positions = np.arange(len(labels))

    ax.bar(
        positions,
        shortlist_losses,
        label="Shortlist loss"
    )

    ax.bar(
        positions,
        ranking_losses,
        bottom=shortlist_losses,
        label="Ranking loss"
    )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)

    ax.set_title(fname)
    ax.set_ylabel("Mean selection gap")
    ax.grid(axis="y",alpha=0.2)
    ax.legend(fontsize=8)

plt.suptitle(
    "Experiment 6C: Why Does Final Selection Fail?\n"
    "Budget = 600"
)

plt.tight_layout()

plt.savefig(
    OUT6C/"error_decomposition.png",
    dpi=160
)

plt.show()

print("Plot saved.")

# ------------------------------------------------------------
# COVERAGE TABLE
# ------------------------------------------------------------

coverage = df6c.groupby(
    ["function","budget","method","fraction"]
)["shortlist_coverage"].mean().reset_index()

print("\nSHORTLIST COVERAGE")
print("="*65)
print(coverage.to_string(index=False))


In [ ]:

# ============================================================
# EXPERIMENT 6D: SHORTLIST-SIZE SENSITIVITY
# Reuses Experiment 6B search trajectories
# No new BO or CMA-ES searches
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

E6D_BUDGETS = [400, 600]
E6D_FRACTIONS = [0.10, 0.25]
E6D_K_VALUES = [1, 5, 10, 20]

E6D_FUNCTIONS = [
    "Sphere", "Ackley", "Rosenbrock", "Rastrigin"
]
E6D_METHODS = ["BO", "CMA-ES"]
E6D_SEEDS = [42, 123]
E6D_NOISE = 0.05
E6D_MAX_SEARCH = 600

# Save directly to Google Drive if mounted.
drive_root = Path(
    "/content/drive/MyDrive/DSA4212_Final_Project"
)

if Path("/content/drive/MyDrive").exists():
    OUT6D = drive_root / "experiment_6d"
else:
    OUT6D = Path("/content/experiment_6d")
    print("WARNING: Drive not mounted. Saving locally.")

OUT6D.mkdir(parents=True, exist_ok=True)

SEARCH6B = Path("/content/experiment_6b/search")

if not SEARCH6B.exists():
    SEARCH6B = (
        drive_root / "experiment_6b/search"
    )

assert SEARCH6B.exists(), (
    "Experiment 6B search files not found. "
    "Restore them from Google Drive."
)

# ------------------------------------------------------------
# DETERMINISTIC VALIDATION NOISE
# ------------------------------------------------------------

def validation_noise(
    seed, fname, method, budget,
    candidate_idx, repeat_idx, sigma
):
    """
    Each candidate gets its own reproducible
    validation-noise sequence.

    For a given candidate, the first r repeated
    observations are identical across shortlist
    policies that allocate at least r repeats.

    Noise is independent across candidates.
    """

    function_id = E6D_FUNCTIONS.index(fname)
    method_id = E6D_METHODS.index(method)

    sequence = np.random.SeedSequence([
        6007,
        int(seed),
        function_id,
        method_id,
        int(budget),
        int(candidate_idx),
        int(repeat_idx)
    ])

    rng = np.random.default_rng(sequence)

    return float(rng.normal(0.0, sigma))

# ------------------------------------------------------------
# RUN ONE SHORTLIST POLICY
# ------------------------------------------------------------

def run_shortlist_policy(
    X, y, true_values,
    fname, method, seed,
    budget, fraction, K, sigma
):

    n_validation = int(round(budget*fraction))
    n_search = budget-n_validation

    assert len(X) == n_search
    assert len(y) == n_search
    assert K <= n_search

    # Only noisy observations determine shortlist.
    shortlist = np.argsort(
        y, kind="stable"
    )[:K]

    sums = y[shortlist].copy()
    counts = np.ones(K, dtype=int)

    # Round-robin validation.
    # Each candidate receives approximately
    # the same number of repeated measurements.
    for j in range(n_validation):

        slot = j % K
        candidate_idx = int(shortlist[slot])

        repeat_idx = int(counts[slot])

        new_noise = validation_noise(
            seed, fname, method, budget,
            candidate_idx, repeat_idx, sigma
        )

        # This simulates a fresh noisy evaluation.
        # The true objective is used only by the
        # measurement simulator, not selection.
        new_observation = (
            true_values[candidate_idx] + new_noise
        )

        sums[slot] += new_observation
        counts[slot] += 1

    estimated_means = sums/counts

    selected_slot = int(
        np.argmin(estimated_means)
    )
    selected_idx = int(
        shortlist[selected_slot]
    )

    # Offline scoring only.
    oracle_idx = int(np.argmin(true_values))
    oracle_regret = float(
        true_values[oracle_idx]
    )

    best_shortlist_regret = float(
        np.min(true_values[shortlist])
    )

    selected_regret = float(
        true_values[selected_idx]
    )

    shortlist_loss = (
        best_shortlist_regret-oracle_regret
    )

    ranking_loss = (
        selected_regret-best_shortlist_regret
    )

    assert np.isclose(
        selected_regret,
        oracle_regret
        + shortlist_loss
        + ranking_loss,
        atol=1e-7
    )

    assert shortlist_loss >= -1e-7
    assert ranking_loss >= -1e-7

    assert n_search+n_validation == budget
    assert counts.sum() == K+n_validation

    return {
        "selected_idx": selected_idx,
        "selected_regret": selected_regret,
        "oracle_regret": oracle_regret,
        "shortlist_loss": shortlist_loss,
        "ranking_loss": ranking_loss,
        "selection_gap":
            selected_regret-oracle_regret,
        "shortlist_coverage":
            int(oracle_idx in shortlist),
        "search_evals": n_search,
        "validation_evals": n_validation,
        "total_evals": budget,
        "min_repeats": int(counts.min()-1),
        "max_repeats": int(counts.max()-1)
    }

# ------------------------------------------------------------
# RUN EXPERIMENT
# ------------------------------------------------------------

records6d = []

total = (
    len(E6D_FUNCTIONS)
    * len(E6D_METHODS)
    * len(E6D_SEEDS)
    * len(E6D_BUDGETS)
    * len(E6D_FRACTIONS)
    * len(E6D_K_VALUES)
)

print("STARTING EXPERIMENT 6D")
print("Expected policy results:", total)
print("=" * 75)

start_time = time.perf_counter()

for fname in E6D_FUNCTIONS:

    func = FUNCTIONS[fname]
    sigma = E6D_NOISE*SCALES[fname]

    for method in E6D_METHODS:

        for seed in E6D_SEEDS:

            filename = (
                f"{fname}_{method}_n{E6D_NOISE:.3f}"
                f"_seed{seed}_B{E6D_MAX_SEARCH}.npz"
            )

            path = SEARCH6B / filename

            if not path.exists():
                raise FileNotFoundError(path)

            with np.load(path) as saved:
                X_full = saved["X"].copy()
                y_full = saved["y"].copy()

            assert len(X_full) == E6D_MAX_SEARCH
            assert len(y_full) == E6D_MAX_SEARCH

            for budget in E6D_BUDGETS:

                for fraction in E6D_FRACTIONS:

                    n_validation = int(
                        round(budget*fraction)
                    )
                    n_search = budget-n_validation

                    X = X_full[:n_search]
                    y = y_full[:n_search]

                    # Used exclusively for simulated
                    # measurement and offline diagnostics.
                    true_values = np.array([
                        func(x) for x in X
                    ])

                    for K in E6D_K_VALUES:

                        result = run_shortlist_policy(
                            X, y, true_values,
                            fname, method, seed,
                            budget, fraction, K, sigma
                        )

                        record = {
                            "function": fname,
                            "method": method,
                            "seed": seed,
                            "noise_fraction": E6D_NOISE,
                            "sigma": sigma,
                            "budget": budget,
                            "fraction": fraction,
                            "shortlist_size": K,
                            **result
                        }

                        records6d.append(record)

# ------------------------------------------------------------
# AUDIT AND SAVE
# ------------------------------------------------------------

df6d = pd.DataFrame(records6d)

assert len(df6d) == total
assert (df6d["total_evals"] == df6d["budget"]).all()
assert (
    df6d["search_evals"]
    + df6d["validation_evals"]
    == df6d["budget"]
).all()

assert (df6d["selection_gap"] >= -1e-7).all()

df6d.to_csv(
    OUT6D / "results.csv",
    index=False
)

metadata6d = {
    "experiment": "6D",
    "purpose": "Shortlist-size sensitivity",
    "budgets": E6D_BUDGETS,
    "fractions": E6D_FRACTIONS,
    "shortlist_sizes": E6D_K_VALUES,
    "functions": E6D_FUNCTIONS,
    "methods": E6D_METHODS,
    "seeds": E6D_SEEDS,
    "noise_fraction": E6D_NOISE,
    "dimension": CFG.d,
    "validation_seed": 6007,
    "validation_policy":
        "Top-K noisy observations, round-robin repeats",
    "noise_coupling":
        "Candidate-specific deterministic streams",
    "note":
        "Exploratory diagnostic. Two seeds only. "
        "Validation noise differs from Experiment 6B."
}

with open(
    OUT6D / "metadata.json", "w"
) as f:
    json.dump(metadata6d, f, indent=2)

print("\nALL AUDIT CHECKS PASSED")
print("Policy results:", len(df6d))
print(
    "Runtime:",
    round(time.perf_counter()-start_time, 2),
    "seconds"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary6d = df6d.groupby(
    [
        "function",
        "budget",
        "method",
        "fraction",
        "shortlist_size"
    ]
).agg(
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    coverage=("shortlist_coverage", "mean"),
    mean_shortlist_loss=("shortlist_loss", "mean"),
    mean_ranking_loss=("ranking_loss", "mean")
).reset_index()

summary6d.to_csv(
    OUT6D / "summary.csv",
    index=False
)

print("\nEXPERIMENT 6D SUMMARY: BUDGET 600")
print("=" * 75)

print(
    summary6d[
        summary6d["budget"] == 600
    ].round(5).to_string(index=False)
)

# ------------------------------------------------------------
# PLOTS: FINAL REGRET VS SHORTLIST SIZE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2, figsize=(13, 9)
)

for ax, fname in zip(
    axes.flat, E6D_FUNCTIONS
):

    subset = summary6d[
        (summary6d["function"] == fname)
        & (summary6d["budget"] == 600)
    ]

    for method in E6D_METHODS:

        for fraction in E6D_FRACTIONS:

            s = subset[
                (subset["method"] == method)
                & np.isclose(
                    subset["fraction"], fraction
                )
            ].sort_values("shortlist_size")

            ax.plot(
                s["shortlist_size"],
                s["mean_regret"],
                marker="o",
                label=f"{method}, {fraction:.0%}"
            )

    ax.set_title(fname)
    ax.set_xlabel("Shortlist size")
    ax.set_ylabel("Mean final true regret")
    ax.set_xticks(E6D_K_VALUES)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle(
    "Experiment 6D: Shortlist-Size Sensitivity\n"
    "Budget = 600, Noise = 5% IQR"
)

plt.tight_layout()
plt.savefig(
    OUT6D / "shortlist_sensitivity.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()

print("\nEXPERIMENT 6D COMPLETE")
print("Saved to:", OUT6D)


In [ ]:

# ============================================================
# EXPERIMENT 7: MAIN MULTI-SEED STUDY
# CPU ONLY | RESUMABLE | SAVES TO GOOGLE DRIVE
# ============================================================

import numpy as np
import pandas as pd
import json
import time
import os
import platform
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import cma
import scipy
import sklearn

# Required earlier definitions:
# CFG, FUNCTIONS, SCALES, initial_design,
# E6Evaluator, e6_run_bo, e6_run_cma

required = [
    "CFG", "FUNCTIONS", "SCALES",
    "initial_design", "E6Evaluator",
    "e6_run_bo", "e6_run_cma"
]

missing = [x for x in required if x not in globals()]
assert not missing, f"Missing definitions: {missing}"

assert CFG.d == 5
assert CFG.n_initial == 10
assert CFG.low == -5.0 and CFG.high == 5.0

ROOT7 = Path(
    "/content/drive/MyDrive/DSA4212_Final_Project/experiment_7"
)

assert Path("/content/drive/MyDrive").exists(), (
    "Mount Google Drive before running."
)

SEARCH7 = ROOT7 / "search"
POLICY7 = ROOT7 / "policies"
SEARCH7.mkdir(parents=True, exist_ok=True)
POLICY7.mkdir(parents=True, exist_ok=True)

B7 = 600
F7 = ["Sphere", "Ackley", "Rosenbrock", "Rastrigin"]
M7 = ["BO", "CMA-ES"]
N7 = [0.05, 0.0]  # Noisy condition first
SEEDS7 = [42, 123] + list(range(1000, 1018))
FRACS7 = [0.0, 0.10, 0.25]
K_VALUES7 = [5, 10]

assert len(SEEDS7) == 20
assert len(set(SEEDS7)) == len(SEEDS7)

CONFIG7 = {
    "experiment": "7",
    "version": "1.0",
    "budget": B7,
    "dimension": CFG.d,
    "functions": F7,
    "methods": M7,
    "noise_fractions": N7,
    "seeds": SEEDS7,
    "validation_fractions": FRACS7,
    "shortlist_sizes": K_VALUES7,
    "initial_points": CFG.n_initial,
    "noise_scales": {k: float(SCALES[k]) for k in F7},
    "bo_policy": "Experiment 6A improved BO",
    "cma_policy": "Experiment 6A CMA-ES",
    "validation": "Top-K noisy observations, equal repeats",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "cma": cma.__version__
}

config_text = json.dumps(CONFIG7, sort_keys=True)
CONFIG_HASH7 = hashlib.sha256(
    config_text.encode()
).hexdigest()[:16]

config_path = ROOT7 / "configuration.json"

if config_path.exists():
    old = json.loads(config_path.read_text())
    assert old == CONFIG7, (
        "Configuration changed. Use a new experiment "
        "folder rather than mixing incompatible runs."
    )
else:
    config_path.write_text(
        json.dumps(CONFIG7, indent=2)
    )

def path7(fname, method, noise, seed):
    return SEARCH7 / (
        f"{fname}_{method}_n{noise:.3f}"
        f"_s{seed}_B{B7}.npz"
    )

def policy_path7(fname, method, noise, seed):
    return POLICY7 / (
        f"{fname}_{method}_n{noise:.3f}"
        f"_s{seed}_B{B7}.csv"
    )

# ------------------------------------------------------------
# SEARCH TRAJECTORY
# ------------------------------------------------------------

def run_search7(fname, method, noise, seed):

    sigma = float(noise * SCALES[fname])

    ev = E6Evaluator(
        FUNCTIONS[fname],
        sigma,
        seed + 100000,
        B7
    )

    start = time.perf_counter()

    if method == "BO":
        e6_run_bo(ev, seed, B7)
    else:
        e6_run_cma(ev, seed, B7)

    elapsed = time.perf_counter() - start

    X = np.asarray(ev.X)
    y = np.asarray(ev.y)

    assert ev.calls == B7
    assert X.shape == (B7, CFG.d)
    assert y.shape == (B7,)
    assert np.isfinite(X).all()
    assert np.isfinite(y).all()
    assert np.all(X >= CFG.low-1e-8)
    assert np.all(X <= CFG.high+1e-8)

    return X, y, elapsed

# ------------------------------------------------------------
# VALIDATION MEASUREMENT
# ------------------------------------------------------------

def noise7(fname, method, noise, seed, idx, rep):

    sequence = np.random.SeedSequence([
        7007,
        F7.index(fname),
        M7.index(method),
        int(round(noise * 100000)),
        int(seed),
        int(idx),
        int(rep)
    ])

    rng = np.random.default_rng(sequence)

    return float(
        rng.normal(0, noise * SCALES[fname])
    )

def validate7(X, y, fname, method, noise,
              seed, fraction, K):

    n_val = int(round(B7 * fraction))
    n_search = B7 - n_val

    Xs = X[:n_search]
    ys = y[:n_search]

    # All shortlist decisions use noisy data only.
    shortlist = np.argsort(
        ys, kind="stable"
    )[:K]

    sums = ys[shortlist].copy()
    counts = np.ones(K, dtype=int)

    func = FUNCTIONS[fname]

    for j in range(n_val):

        slot = j % K
        idx = int(shortlist[slot])
        repeat_number = int(counts[slot])

        # Fresh independent noisy evaluation.
        measurement = (
            func(Xs[idx])
            + noise7(
                fname, method, noise,
                seed, idx, repeat_number
            )
        )

        sums[slot] += measurement
        counts[slot] += 1

    selected_slot = int(np.argmin(sums/counts))
    selected_idx = int(shortlist[selected_slot])

    # Independent offline scoring.
    true_values = np.array([
        func(x) for x in Xs
    ])

    oracle = float(np.min(true_values))
    best_shortlist = float(
        np.min(true_values[shortlist])
    )
    selected = float(true_values[selected_idx])

    shortlist_loss = best_shortlist - oracle
    ranking_loss = selected - best_shortlist

    assert np.isclose(
        selected,
        oracle + shortlist_loss + ranking_loss,
        atol=1e-6
    )
    assert shortlist_loss >= -1e-6
    assert ranking_loss >= -1e-6
    assert n_search + n_val == B7

    return {
        "function": fname,
        "method": method,
        "noise": noise,
        "sigma": noise * SCALES[fname],
        "seed": seed,
        "budget": B7,
        "fraction": fraction,
        "K": K,
        "search_evals": n_search,
        "validation_evals": n_val,
        "total_evals": B7,
        "selected_idx": selected_idx,
        "selected_regret": selected,
        "oracle_regret": oracle,
        "shortlist_loss": shortlist_loss,
        "ranking_loss": ranking_loss,
        "selection_gap": selected - oracle,
        "shortlist_coverage": int(
            np.argmin(true_values) in shortlist
        ),
        "config_hash": CONFIG_HASH7
    }

print("EXPERIMENT 7 FRAMEWORK READY")
print("Configuration:", CONFIG_HASH7)
print("Saving to:", ROOT7)


In [ ]:

# ============================================================
# EXPERIMENT 7: RESUMABLE MAIN EXECUTION
# ============================================================

import traceback

total_searches = (
    len(F7) * len(M7) * len(N7) * len(SEEDS7)
)

completed = 0
new_searches = 0
failures = []

print("STARTING EXPERIMENT 7")
print("Expected search trajectories:", total_searches)
print("Output:", ROOT7)
print("=" * 75, flush=True)

start_all = time.perf_counter()

# Prioritise noisy experiments and BO.
for noise in N7:
    for fname in F7:
        for method in M7:
            for seed in SEEDS7:

                search_file = path7(
                    fname, method, noise, seed
                )
                policy_file = policy_path7(
                    fname, method, noise, seed
                )

                try:
                    if search_file.exists():
                        with np.load(
                            search_file,
                            allow_pickle=False
                        ) as saved:
                            X = saved["X"].copy()
                            y = saved["y"].copy()
                            elapsed = float(saved["elapsed"])
                            saved_hash = str(
                                saved["config_hash"].item()
                            )

                        assert saved_hash == CONFIG_HASH7
                        assert X.shape == (B7, CFG.d)
                        assert y.shape == (B7,)
                        assert np.isfinite(X).all()
                        assert np.isfinite(y).all()
                        status = "LOADED"

                    else:
                        X, y, elapsed = run_search7(
                            fname, method, noise, seed
                        )

                        # Temporary file then atomic rename.
                        temp_file = search_file.with_suffix(
                            ".tmp.npz"
                        )

                        np.savez_compressed(
                            temp_file,
                            X=X,
                            y=y,
                            elapsed=elapsed,
                            config_hash=CONFIG_HASH7
                        )

                        os.replace(
                            temp_file, search_file
                        )

                        new_searches += 1
                        status = "NEW"

                    # Policy results can be recreated cheaply.
                    if policy_file.exists():
                        existing = pd.read_csv(policy_file)
                        assert len(existing) == 5
                        assert (
                            existing["config_hash"]
                            == CONFIG_HASH7
                        ).all()
                    else:
                        policy_records = []

                        for fraction in FRACS7:

                            # K has no effect when validation=0.
                            ks = (
                                [5] if fraction == 0
                                else K_VALUES7
                            )

                            for K in ks:
                                row = validate7(
                                    X, y, fname, method,
                                    noise, seed, fraction, K
                                )
                                row["search_full_runtime"] = elapsed
                                policy_records.append(row)

                        temp_csv = policy_file.with_suffix(
                            ".tmp.csv"
                        )

                        pd.DataFrame(
                            policy_records
                        ).to_csv(temp_csv, index=False)

                        os.replace(temp_csv, policy_file)

                    completed += 1

                    print(
                        f"{completed:03d}/{total_searches} "
                        f"{fname:12s} {method:6s} "
                        f"noise={noise:.2f} seed={seed} "
                        f"time={elapsed:.1f}s [{status}]",
                        flush=True
                    )

                except Exception as exc:
                    failure = {
                        "function": fname,
                        "method": method,
                        "noise": noise,
                        "seed": seed,
                        "error": repr(exc),
                        "traceback": traceback.format_exc()
                    }

                    failures.append(failure)

                    (ROOT7 / "failures.json").write_text(
                        json.dumps(failures, indent=2)
                    )

                    print(
                        "FAILED:",
                        fname, method, noise, seed,
                        repr(exc),
                        flush=True
                    )

                    # Stop on unexpected errors.
                    # We should inspect them rather than
                    # silently continue with invalid results.
                    raise

print("\n" + "=" * 75)
print("EXPERIMENT 7 EXECUTION COMPLETE")
print("Completed:", completed)
print("New trajectories:", new_searches)
print("Elapsed hours:",
      round((time.perf_counter()-start_all)/3600, 2))
print("Output:", ROOT7)


In [ ]:

# ============================================================
# EXPERIMENT 7: SUMMARY AND STATISTICAL ANALYSIS
# Run after overnight execution
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

files = sorted(POLICY7.glob("*.csv"))
expected_files = 320

print("Completed trajectories:",
      len(files), "/", expected_files)

assert len(files) == expected_files, (
    "Study incomplete. Resume Cell 2 first."
)

df7 = pd.concat(
    [pd.read_csv(p) for p in files],
    ignore_index=True
)

assert len(df7) == 1600
assert (df7["total_evals"] == B7).all()
assert (df7["config_hash"] == CONFIG_HASH7).all()
assert (
    df7["search_evals"] + df7["validation_evals"]
    == B7
).all()

df7.to_csv(ROOT7 / "all_results.csv", index=False)

# Primary analysis: K=5, including no-validation.
primary = df7[df7["K"] == 5].copy()

summary = primary.groupby(
    ["function", "noise", "method", "fraction"]
).agg(
    n=("selected_regret", "count"),
    mean_regret=("selected_regret", "mean"),
    median_regret=("selected_regret", "median"),
    std_regret=("selected_regret", "std"),
    mean_oracle=("oracle_regret", "mean"),
    mean_gap=("selection_gap", "mean"),
    coverage=("shortlist_coverage", "mean")
).reset_index()

summary.to_csv(
    ROOT7 / "primary_summary.csv", index=False
)

print("\nPRIMARY SUMMARY")
print(summary.round(5).to_string(index=False))

# Paired bootstrap differences:
# Difference = validation regret - no-validation regret.
# Negative values favour validation.

rng = np.random.default_rng(777)
comparison = []

for (fname, noise, method), group in primary.groupby(
    ["function", "noise", "method"]
):

    pivot = group.pivot(
        index="seed",
        columns="fraction",
        values="selected_regret"
    )

    assert len(pivot) == 20
    assert not pivot.isna().any().any()

    for fraction in [0.10, 0.25]:

        differences = (
            pivot[fraction] - pivot[0.0]
        ).to_numpy()

        boot = rng.choice(
            differences,
            size=(10000, len(differences)),
            replace=True
        ).mean(axis=1)

        lo, hi = np.percentile(
            boot, [2.5, 97.5]
        )

        comparison.append({
            "function": fname,
            "noise": noise,
            "method": method,
            "fraction": fraction,
            "mean_paired_difference":
                float(np.mean(differences)),
            "ci_lower": float(lo),
            "ci_upper": float(hi),
            "n_seeds": len(differences)
        })

comparison = pd.DataFrame(comparison)

comparison.to_csv(
    ROOT7 / "paired_bootstrap.csv",
    index=False
)

print("\nPAIRED BOOTSTRAP COMPARISONS")
print(comparison.round(5).to_string(index=False))

# Main figures, saved to Google Drive.
for noise in N7:

    fig, axes = plt.subplots(
        2, 2, figsize=(12, 9)
    )

    for ax, fname in zip(axes.flat, F7):

        for method in M7:
            s = summary[
                (summary["function"] == fname)
                & (summary["noise"] == noise)
                & (summary["method"] == method)
            ].sort_values("fraction")

            ax.plot(
                s["fraction"] * 100,
                s["mean_regret"],
                marker="o",
                label=method
            )

        ax.set_title(fname)
        ax.set_xlabel("Validation budget (%)")
        ax.set_ylabel("Mean true simple regret")
        ax.grid(alpha=0.3)
        ax.legend()

    fig.suptitle(
        f"Experiment 7: Noise fraction = {noise}"
    )
    fig.tight_layout()

    fig.savefig(
        ROOT7 / f"main_regret_noise_{noise:.2f}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

print("\nANALYSIS COMPLETE")
print("Results and figures saved to:", ROOT7)


## Original 8A (superseded; retained as exploratory history)


In [ ]:

# ============================================================
# EXPERIMENT 8A: CMA-ES POPULATION-SIZE SENSITIVITY
# Fixed 600-evaluation budget, paired seeds
# ============================================================

import numpy as np
import pandas as pd
import cma
import json
import time
from pathlib import Path

ROOT8A = Path(
    "/content/drive/MyDrive/DSA4212_Final_Project/experiment_8a"
)
ROOT8A.mkdir(parents=True, exist_ok=True)

B8A = 600
F8A = ["Sphere", "Ackley", "Rosenbrock", "Rastrigin"]
N8A = [0.0, 0.05]
SEEDS8A = [42, 123] + list(range(1000, 1018))
FRACTIONS8A = [0.0, 0.10, 0.25]
K8A = 5

# Get the CMA-ES default population size for dimension 5.
probe = cma.CMAEvolutionStrategy(
    np.zeros(CFG.d),
    2.0,
    {"seed": 42, "verbose": -9}
)
DEFAULT_POP = int(probe.popsize)

POP_SIZES8A = sorted(set([
    max(2, DEFAULT_POP // 2),
    DEFAULT_POP,
    DEFAULT_POP * 2
]))

print("Default population:", DEFAULT_POP)
print("Population sizes:", POP_SIZES8A)

def run_cma8a(fname, noise, seed, popsize):
    sigma_noise = noise * SCALES[fname]
    rng = np.random.default_rng(seed + 100000)

    # Use a reproducible initial mean.
    x0 = np.random.default_rng(seed).uniform(
        CFG.low, CFG.high, size=CFG.d
    )

    es = cma.CMAEvolutionStrategy(
        x0,
        2.0,
        {
            "bounds": [CFG.low, CFG.high],
            "popsize": int(popsize),
            "seed": int(seed),
            "verbose": -9
        }
    )

    X, y = [], []
    start = time.perf_counter()

    while len(y) < B8A:
        remaining = B8A - len(y)
        candidates = es.ask()

        evaluated_x = []
        evaluated_y = []

        for x in candidates[:remaining]:
            x = np.clip(
                np.asarray(x),
                CFG.low, CFG.high
            )

            value = (
                FUNCTIONS[fname](x)
                + rng.normal(0, sigma_noise)
            )

            X.append(x.copy())
            y.append(float(value))
            evaluated_x.append(x)
            evaluated_y.append(float(value))

        # Only update with complete generations.
        if len(evaluated_x) == len(candidates):
            es.tell(evaluated_x, evaluated_y)

    elapsed = time.perf_counter() - start

    assert len(y) == B8A
    return np.asarray(X), np.asarray(y), elapsed

def validate8a(X, y, fname, noise, seed, fraction, K):
    n_val = int(round(B8A * fraction))
    n_search = B8A - n_val

    Xs, ys = X[:n_search], y[:n_search]
    shortlist = np.argsort(
        ys, kind="stable"
    )[:K]

    sums = ys[shortlist].copy()
    counts = np.ones(K, dtype=int)

    for j in range(n_val):
        slot = j % K
        idx = int(shortlist[slot])

        rng = np.random.default_rng(
            np.random.SeedSequence([
                8008, seed, F8A.index(fname),
                int(noise * 100000),
                idx, int(counts[slot])
            ])
        )

        measurement = (
            FUNCTIONS[fname](Xs[idx])
            + rng.normal(0, noise * SCALES[fname])
        )

        sums[slot] += measurement
        counts[slot] += 1

    selected_idx = int(
        shortlist[np.argmin(sums/counts)]
    )

    true_values = np.array([
        FUNCTIONS[fname](x) for x in Xs
    ])

    selected = float(true_values[selected_idx])
    oracle = float(np.min(true_values))

    return {
        "selected_regret": selected,
        "oracle_regret": oracle,
        "selection_gap": selected-oracle,
        "search_evals": n_search,
        "validation_evals": n_val,
        "total_evals": B8A
    }

records8a = []
total = (
    len(F8A) * len(N8A)
    * len(SEEDS8A) * len(POP_SIZES8A)
)

print("\nSTARTING EXPERIMENT 8A")
print("Search trajectories:", total)
print("=" * 70)

counter = 0

for fname in F8A:
    for noise in N8A:
        for seed in SEEDS8A:
            for popsize in POP_SIZES8A:

                counter += 1

                X, y, elapsed = run_cma8a(
                    fname, noise, seed, popsize
                )

                for fraction in FRACTIONS8A:
                    result = validate8a(
                        X, y, fname, noise,
                        seed, fraction, K8A
                    )

                    records8a.append({
                        "function": fname,
                        "noise": noise,
                        "seed": seed,
                        "popsize": popsize,
                        "fraction": fraction,
                        "runtime": elapsed,
                        **result
                    })

                print(
                    f"{counter:03d}/{total} "
                    f"{fname:12s} noise={noise:.2f} "
                    f"seed={seed} pop={popsize} "
                    f"time={elapsed:.3f}s",
                    flush=True
                )

df8a = pd.DataFrame(records8a)

assert (
    df8a["search_evals"]
    + df8a["validation_evals"]
    == B8A
).all()

df8a.to_csv(
    ROOT8A / "results.csv",
    index=False
)

summary8a = df8a.groupby(
    ["function", "noise", "popsize", "fraction"]
).agg(
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    mean_gap=("selection_gap", "mean")
).reset_index()

summary8a.to_csv(
    ROOT8A / "summary.csv",
    index=False
)

print("\nEXPERIMENT 8A SUMMARY")
print(summary8a.round(5).to_string(index=False))
print("\nSaved to:", ROOT8A)


In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, fname in zip(axes.flat, F8A):
    subset = summary8a[
        (summary8a["function"] == fname)
        & (summary8a["noise"] == 0.05)
    ]

    for fraction in FRACTIONS8A:
        s = subset[
            np.isclose(subset["fraction"], fraction)
        ].sort_values("popsize")

        ax.plot(
            s["popsize"],
            s["mean_regret"],
            marker="o",
            label=f"Validation {fraction:.0%}"
        )

    ax.set_title(fname)
    ax.set_xlabel("CMA-ES population size")
    ax.set_ylabel("Mean true regret")
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle(
    "Experiment 8A: CMA-ES Population Sensitivity"
)
plt.tight_layout()

plt.savefig(
    ROOT8A / "population_sensitivity.png",
    dpi=200
)
plt.show()


## Original 8B (superseded; retained as exploratory history)


In [ ]:

# ============================================================
# EXPERIMENT 8B
# TOP-K NOISY VS DIVERSITY-AWARE SHORTLISTS
# Reuses Experiment 7 trajectories
# ============================================================

import numpy as np
import pandas as pd
import json
from pathlib import Path
from scipy.spatial.distance import cdist

ROOT8B = Path(
    "/content/drive/MyDrive/"
    "DSA4212_Final_Project/experiment_8b"
)
ROOT8B.mkdir(parents=True, exist_ok=True)

SEARCH8B = Path(
    "/content/drive/MyDrive/"
    "DSA4212_Final_Project/experiment_7/search"
)

assert SEARCH8B.exists()

F8B = ["Rosenbrock", "Rastrigin"]
M8B = ["BO", "CMA-ES"]
SEEDS8B = [42, 123] + list(range(1000, 1018))

B8B = 600
NOISE8B = 0.05
FRACS8B = [0.10, 0.25]
K8B = 5

STRATEGIES8B = [
    "top_noisy",
    "diverse"
]

# Diversity is measured in normalized input space.
# We use a greedy selection rule with a fixed
# minimum separation threshold.
MIN_DIST8B = 0.15

def shortlist8b(X, y, strategy, K):

    order = np.argsort(y, kind="stable")

    if strategy == "top_noisy":
        return order[:K]

    if strategy != "diverse":
        raise ValueError(strategy)

    # Normalize coordinates to [0, 1].
    Xn = (X-CFG.low)/(CFG.high-CFG.low)

    selected = []

    for idx in order:

        idx = int(idx)

        if not selected:
            selected.append(idx)
            continue

        distances = np.linalg.norm(
            Xn[selected]-Xn[idx],
            axis=1
        )

        if np.min(distances) >= MIN_DIST8B:
            selected.append(idx)

        if len(selected) == K:
            break

    # If the threshold prevents finding K points,
    # fill remaining slots by noisy ranking.
    if len(selected) < K:
        for idx in order:
            idx = int(idx)
            if idx not in selected:
                selected.append(idx)
            if len(selected) == K:
                break

    assert len(selected) == K
    assert len(set(selected)) == K

    return np.array(selected, dtype=int)

def validation_noise8b(
    fname, method, seed, idx, rep, sigma
):

    rng = np.random.default_rng(
        np.random.SeedSequence([
            8009,
            F8B.index(fname),
            M8B.index(method),
            int(seed),
            int(idx),
            int(rep)
        ])
    )

    return float(rng.normal(0, sigma))

def evaluate_policy8b(
    X, y, fname, method,
    seed, fraction, strategy
):

    n_val = int(round(B8B*fraction))
    n_search = B8B-n_val

    Xs = X[:n_search]
    ys = y[:n_search]

    shortlist = shortlist8b(
        Xs, ys, strategy, K8B
    )

    sigma = NOISE8B*SCALES[fname]
    func = FUNCTIONS[fname]

    sums = ys[shortlist].copy()
    counts = np.ones(K8B, dtype=int)

    for j in range(n_val):

        slot = j % K8B
        idx = int(shortlist[slot])
        rep = int(counts[slot])

        obs = (
            func(Xs[idx])
            + validation_noise8b(
                fname, method, seed,
                idx, rep, sigma
            )
        )

        sums[slot] += obs
        counts[slot] += 1

    selected_idx = int(
        shortlist[np.argmin(sums/counts)]
    )

    # True values used only for offline scoring.
    true_values = np.array([
        func(x) for x in Xs
    ])

    oracle_idx = int(np.argmin(true_values))
    oracle = float(true_values[oracle_idx])

    shortlist_best = float(
        np.min(true_values[shortlist])
    )

    selected = float(
        true_values[selected_idx]
    )

    shortlist_loss = shortlist_best-oracle
    ranking_loss = selected-shortlist_best

    assert np.isclose(
        selected,
        oracle+shortlist_loss+ranking_loss,
        atol=1e-6
    )

    assert n_search+n_val == B8B

    return {
        "selected_regret": selected,
        "oracle_regret": oracle,
        "shortlist_loss": shortlist_loss,
        "ranking_loss": ranking_loss,
        "selection_gap": selected-oracle,
        "coverage": int(oracle_idx in shortlist),
        "selected_idx": selected_idx,
        "search_evals": n_search,
        "validation_evals": n_val,
        "total_evals": B8B
    }

records8b = []

expected = (
    len(F8B)*len(M8B)*len(SEEDS8B)
    * len(FRACS8B)*len(STRATEGIES8B)
)

print("STARTING EXPERIMENT 8B")
print("Expected policies:", expected)
print("=" * 70)

for fname in F8B:
    for method in M8B:
        for seed in SEEDS8B:

            filename = (
                f"{fname}_{method}_n{NOISE8B:.3f}"
                f"_s{seed}_B{B8B}.npz"
            )

            path = SEARCH8B / filename
            assert path.exists(), path

            with np.load(path, allow_pickle=False) as saved:
                X = saved["X"].copy()
                y = saved["y"].copy()

            assert X.shape == (B8B, CFG.d)
            assert y.shape == (B8B,)

            for fraction in FRACS8B:
                for strategy in STRATEGIES8B:

                    result = evaluate_policy8b(
                        X, y, fname, method,
                        seed, fraction, strategy
                    )

                    records8b.append({
                        "function": fname,
                        "method": method,
                        "seed": seed,
                        "noise": NOISE8B,
                        "fraction": fraction,
                        "strategy": strategy,
                        **result
                    })

df8b = pd.DataFrame(records8b)

assert len(df8b) == expected
assert (df8b["total_evals"] == B8B).all()
assert (df8b["shortlist_loss"] >= -1e-6).all()
assert (df8b["ranking_loss"] >= -1e-6).all()

df8b.to_csv(
    ROOT8B/"results.csv",
    index=False
)

summary8b = df8b.groupby(
    ["function","method","fraction","strategy"]
).agg(
    mean_regret=("selected_regret","mean"),
    std_regret=("selected_regret","std"),
    mean_oracle=("oracle_regret","mean"),
    mean_shortlist_loss=("shortlist_loss","mean"),
    mean_ranking_loss=("ranking_loss","mean"),
    coverage=("coverage","mean")
).reset_index()

summary8b.to_csv(
    ROOT8B/"summary.csv",
    index=False
)

print("\nEXPERIMENT 8B SUMMARY")
print(summary8b.round(5).to_string(index=False))
print("\nSaved to:", ROOT8B)


In [ ]:

# ============================================================
# EXPERIMENT 8B ANALYSIS
# ============================================================

import matplotlib.pyplot as plt

rng = np.random.default_rng(8123)
comparisons = []

for (fname, method, fraction), group in df8b.groupby(
    ["function","method","fraction"]
):

    pivot = group.pivot(
        index="seed",
        columns="strategy",
        values="selected_regret"
    )

    diff = (
        pivot["diverse"]-pivot["top_noisy"]
    ).to_numpy()

    bootstrap = rng.choice(
        diff,
        size=(10000,len(diff)),
        replace=True
    ).mean(axis=1)

    lo, hi = np.percentile(
        bootstrap,[2.5,97.5]
    )

    comparisons.append({
        "function":fname,
        "method":method,
        "fraction":fraction,
        "mean_difference":np.mean(diff),
        "ci_lower":lo,
        "ci_upper":hi
    })

comparison8b = pd.DataFrame(comparisons)

comparison8b.to_csv(
    ROOT8B/"paired_comparisons.csv",
    index=False
)

print("PAIRED COMPARISONS")
print(comparison8b.round(5).to_string(index=False))

fig, axes = plt.subplots(
    1,2,figsize=(12,5)
)

for ax, fname in zip(axes,F8B):

    subset = summary8b[
        summary8b["function"]==fname
    ]

    for method in M8B:
        for strategy in STRATEGIES8B:

            s = subset[
                (subset["method"]==method)
                & (subset["strategy"]==strategy)
            ].sort_values("fraction")

            ax.plot(
                s["fraction"]*100,
                s["mean_regret"],
                marker="o",
                label=f"{method}, {strategy}"
            )

    ax.set_title(fname)
    ax.set_xlabel("Validation budget (%)")
    ax.set_ylabel("Mean final true regret")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle(
    "Experiment 8B: Does Diversity Improve Selection?"
)
plt.tight_layout()

plt.savefig(
    ROOT8B/"diversity_comparison.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

print("\nEXPERIMENT 8B COMPLETE")


In [ ]:

# EXPERIMENT 7: SAVED DATA INTEGRITY CHECK

from pathlib import Path
import numpy as np
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone

ROOT = Path(
    "/content/drive/MyDrive/DSA4212_Final_Project"
)

SEARCH = ROOT / "experiment_7/search"
assert SEARCH.exists()

files = sorted(SEARCH.glob("*.npz"))

assert len(files) == 320, (
    f"Expected 320 trajectories, found {len(files)}"
)

records = []

for path in files:
    with np.load(path, allow_pickle=False) as data:
        X = data["X"]
        y = data["y"]

        assert X.shape == (600, 5)
        assert y.shape == (600,)
        assert np.isfinite(X).all()
        assert np.isfinite(y).all()

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    records.append({
        "filename": path.name,
        "sha256": digest,
        "evaluations": len(y),
        "dimension": X.shape[1]
    })

manifest_path = ROOT / "experiment_7/trajectory_manifest.csv"
manifest = pd.DataFrame(records)

if manifest_path.exists():
    previous = pd.read_csv(manifest_path)
    merged = previous.merge(
        manifest,
        on="filename",
        suffixes=("_previous", "_current"),
        validate="one_to_one"
    )
    assert len(merged) == 320
    changed = merged[
        merged["sha256_previous"]
        != merged["sha256_current"]
    ]
    print("Changed files:", len(changed))
    if len(changed):
        print(changed[["filename"]].to_string(index=False))
    else:
        print("PASS: All saved trajectory hashes unchanged")
else:
    manifest.to_csv(manifest_path, index=False)
    print("Created baseline trajectory manifest.")

print("PASS: 320 trajectories structurally valid")
print("Manifest:", manifest_path)


In [ ]:

# ============================================================
# EXPERIMENT 8C: NOISE-LEVEL SENSITIVITY
# 1% IQR noise, new paired seeds
# Reuses Experiment 7 implementations
# ============================================================

import numpy as np
import pandas as pd
import json
import os
import time
import hashlib
import traceback
from pathlib import Path

ROOT8C = Path(
    "/content/drive/MyDrive/"
    "DSA4212_Final_Project/experiment_8c"
)

SEARCH8C = ROOT8C / "search"
POLICY8C = ROOT8C / "policies"

SEARCH8C.mkdir(parents=True, exist_ok=True)
POLICY8C.mkdir(parents=True, exist_ok=True)

F8C = ["Rosenbrock", "Rastrigin"]
M8C = ["BO", "CMA-ES"]
NOISE8C = 0.01
SEEDS8C = list(range(3000, 3010))
FRACS8C = [0.0, 0.10, 0.25]
K8C = 5
B8C = 600

assert len(SEEDS8C) == 10
assert CFG.d == 5
assert B7 == B8C

required = [
    "run_search7", "validate7",
    "FUNCTIONS", "SCALES", "CFG"
]

missing = [
    name for name in required
    if name not in globals()
]

assert not missing, (
    f"Missing Experiment 7 definitions: {missing}"
)

# Experiment 7 functions use global B7,
# which remains fixed at 600.
assert B7 == 600

CONFIG8C = {
    "experiment": "8C",
    "purpose": "Intermediate noise sensitivity",
    "functions": F8C,
    "methods": M8C,
    "budget": B8C,
    "dimension": CFG.d,
    "noise_fraction": NOISE8C,
    "seeds": SEEDS8C,
    "validation_fractions": FRACS8C,
    "shortlist_size": K8C,
    "search_implementation": "Experiment 7",
    "validation_implementation": "Experiment 7",
    "noise_scales": {
        f: float(SCALES[f]) for f in F8C
    }
}

config_file = ROOT8C / "configuration.json"

if config_file.exists():
    previous = json.loads(config_file.read_text())
    assert previous == CONFIG8C, (
        "Configuration mismatch: do not mix versions."
    )
else:
    config_file.write_text(
        json.dumps(CONFIG8C, indent=2)
    )

CONFIG_HASH8C = hashlib.sha256(
    json.dumps(CONFIG8C, sort_keys=True).encode()
).hexdigest()[:16]

def search_path8c(fname, method, seed):
    return SEARCH8C / (
        f"{fname}_{method}_n{NOISE8C:.3f}"
        f"_s{seed}_B{B8C}.npz"
    )

def policy_path8c(fname, method, seed):
    return POLICY8C / (
        f"{fname}_{method}_n{NOISE8C:.3f}"
        f"_s{seed}_B{B8C}.csv"
    )

total8c = (
    len(F8C)*len(M8C)*len(SEEDS8C)
)

completed8c = 0
print("STARTING EXPERIMENT 8C")
print("Expected search trajectories:", total8c)
print("Noise fraction:", NOISE8C)
print("Saving to:", ROOT8C)
print("=" * 75)

start8c = time.perf_counter()

for fname in F8C:
    for method in M8C:
        for seed in SEEDS8C:

            path = search_path8c(
                fname, method, seed
            )

            policy_path = policy_path8c(
                fname, method, seed
            )

            if path.exists():

                with np.load(
                    path, allow_pickle=False
                ) as saved:
                    X = saved["X"].copy()
                    y = saved["y"].copy()
                    elapsed = float(saved["elapsed"])
                    saved_hash = str(
                        saved["config_hash"].item()
                    )

                assert saved_hash == CONFIG_HASH8C
                assert X.shape == (B8C, CFG.d)
                assert y.shape == (B8C,)
                assert np.isfinite(X).all()
                assert np.isfinite(y).all()

                status = "LOADED"

            else:

                # Reuses Experiment 7 search runner.
                X, y, elapsed = run_search7(
                    fname, method, NOISE8C, seed
                )

                temp = path.with_suffix(".tmp.npz")

                np.savez_compressed(
                    temp,
                    X=X,
                    y=y,
                    elapsed=elapsed,
                    config_hash=CONFIG_HASH8C
                )

                os.replace(temp, path)
                status = "NEW"

            # Recompute policies from saved search.
            # This uses exactly the Experiment 7
            # validation procedure.
            records = []

            for fraction in FRACS8C:

                result = validate7(
                    X, y,
                    fname, method,
                    NOISE8C, seed,
                    fraction, K8C
                )

                records.append({
                    **result,
                    "experiment": "8C",
                    "config_hash_8c": CONFIG_HASH8C,
                    "search_runtime": elapsed
                })

            pd.DataFrame(records).to_csv(
                policy_path,
                index=False
            )

            completed8c += 1

            print(
                f"{completed8c:02d}/{total8c} "
                f"{fname:12s} {method:6s} "
                f"seed={seed} "
                f"time={elapsed:.2f}s "
                f"[{status}]",
                flush=True
            )

print("\nEXPERIMENT 8C SEARCH COMPLETE")
print("Trajectories:", completed8c)
print("Runtime (minutes):",
      round((time.perf_counter()-start8c)/60, 2))


In [ ]:

# ============================================================
# EXPERIMENT 8C: ANALYSIS
# Paired bootstrap confidence intervals
# ============================================================

import matplotlib.pyplot as plt

files8c = sorted(POLICY8C.glob("*.csv"))

assert len(files8c) == 40, (
    f"Expected 40 policy files, found {len(files8c)}"
)

df8c = pd.concat(
    [pd.read_csv(p) for p in files8c],
    ignore_index=True
)

assert len(df8c) == 120
assert (df8c["total_evals"] == B8C).all()
assert (
    df8c["search_evals"]
    + df8c["validation_evals"]
    == B8C
).all()

df8c.to_csv(
    ROOT8C / "all_results.csv",
    index=False
)

summary8c = df8c.groupby(
    ["function", "method", "fraction"]
).agg(
    n=("selected_regret", "count"),
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    mean_oracle=("oracle_regret", "mean"),
    mean_gap=("selection_gap", "mean"),
    coverage=("shortlist_coverage", "mean")
).reset_index()

summary8c.to_csv(
    ROOT8C / "summary.csv",
    index=False
)

print("EXPERIMENT 8C SUMMARY")
print("=" * 75)
print(summary8c.round(5).to_string(index=False))

# Paired bootstrap
rng = np.random.default_rng(8080)
comparisons8c = []

for (fname, method), group in df8c.groupby(
    ["function", "method"]
):

    pivot = group.pivot(
        index="seed",
        columns="fraction",
        values="selected_regret"
    )

    assert len(pivot) == 10

    for fraction in [0.10, 0.25]:

        differences = (
            pivot[fraction] - pivot[0.0]
        ).to_numpy()

        boot = rng.choice(
            differences,
            size=(10000, len(differences)),
            replace=True
        ).mean(axis=1)

        lo, hi = np.percentile(
            boot, [2.5, 97.5]
        )

        comparisons8c.append({
            "function": fname,
            "method": method,
            "fraction": fraction,
            "mean_difference":
                float(differences.mean()),
            "ci_lower": float(lo),
            "ci_upper": float(hi),
            "n_seeds": len(differences)
        })

comparison8c = pd.DataFrame(comparisons8c)

comparison8c.to_csv(
    ROOT8C / "paired_bootstrap.csv",
    index=False
)

print("\nPAIRED BOOTSTRAP COMPARISONS")
print("=" * 75)
print(
    comparison8c.round(5).to_string(index=False)
)

fig, axes = plt.subplots(
    1, 2, figsize=(12, 5)
)

for ax, fname in zip(axes, F8C):

    for method in M8C:

        s = summary8c[
            (summary8c["function"] == fname)
            & (summary8c["method"] == method)
        ].sort_values("fraction")

        ax.plot(
            s["fraction"] * 100,
            s["mean_regret"],
            marker="o",
            label=method
        )

    ax.set_title(fname)
    ax.set_xlabel("Validation allocation (%)")
    ax.set_ylabel("Mean final true regret")
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle(
    "Experiment 8C: Intermediate Noise (1% IQR)"
)

plt.tight_layout()

plt.savefig(
    ROOT8C / "noise_sensitivity.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("\nEXPERIMENT 8C COMPLETE")
print("Saved to:", ROOT8C)


## Corrected 8A-R (authoritative population-size sensitivity)


In [ ]:

# ============================================================
# EXPERIMENT 8A-R: CORRECTED CMA-ES SENSITIVITY
# Matches Experiment 7 CMA-ES implementation
# Changes ONLY population size
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cma
import time
import json
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/DSA4212_Final_Project"
)
OUT = ROOT / "experiment_8a_corrected"
OUT.mkdir(parents=True, exist_ok=True)

assert (ROOT / "experiment_7/search").exists()
assert CFG.d == 5
assert CFG.n_initial == 10

FUNCTION_NAMES = [
    "Sphere", "Ackley", "Rosenbrock", "Rastrigin"
]
NOISE_LEVELS = [0.0, 0.05]
SEEDS = [42, 123] + list(range(1000, 1018))
POP_SIZES = [4, 8, 16]
BUDGET = 600
FRACTIONS = [0.0, 0.10, 0.25]
K = 5

# ------------------------------------------------------------
# Exact Experiment 7 CMA-ES logic, with popsize parameter
# ------------------------------------------------------------

def corrected_cma_run(fname, noise, seed, popsize):

    sigma = noise * SCALES[fname]

    ev = E6Evaluator(
        FUNCTIONS[fname],
        sigma,
        seed + 100000,
        BUDGET
    )

    X0, y0 = e6_initialise(ev, seed)

    best_x = X0[np.argmin(y0)].copy()
    best_y = float(np.min(y0))

    def make_es(center, es_seed):
        return cma.CMAEvolutionStrategy(
            np.clip(center, CFG.low, CFG.high),
            2.0,
            {
                "seed": int(es_seed),
                "bounds": [CFG.low, CFG.high],
                "popsize": int(popsize),
                "verbose": -9,
                "verb_disp": 0,
                "tolfun": 0,
                "tolx": 1e-14,
                "tolstagnation": 100000
            }
        )

    es = make_es(best_x, seed)

    start = time.perf_counter()

    while ev.calls < BUDGET:

        candidates = es.ask()
        remaining = BUDGET - ev.calls
        batch = candidates[:remaining]

        values = []

        for x in batch:
            value = ev.evaluate(x, "search")
            values.append(value)

            if value < best_y:
                best_y = value
                best_x = np.asarray(x).copy()

        if len(batch) == len(candidates):
            es.tell(candidates, values)
        else:
            break

        if es.stop():
            es = make_es(
                best_x,
                seed + ev.calls + 1
            )

    assert ev.calls == BUDGET

    return np.asarray(ev.X), np.asarray(ev.y), (
        time.perf_counter() - start
    )

# ------------------------------------------------------------
# Validation: same policy as Experiment 7
# ------------------------------------------------------------

def corrected_policy(X, y, fname, noise,
                     seed, fraction):

    n_val = int(round(BUDGET * fraction))
    n_search = BUDGET - n_val

    Xs = X[:n_search]
    ys = y[:n_search]

    shortlist = np.argsort(
        ys, kind="stable"
    )[:K]

    sums = ys[shortlist].copy()
    counts = np.ones(K, dtype=int)

    for j in range(n_val):

        slot = j % K
        idx = int(shortlist[slot])
        rep = int(counts[slot])

        # Use the existing Experiment 7
        # deterministic validation-noise function.
        new_y = (
            FUNCTIONS[fname](Xs[idx])
            + noise7(
                fname, "CMA-ES",
                noise, seed, idx, rep
            )
        )

        sums[slot] += new_y
        counts[slot] += 1

    selected_idx = int(
        shortlist[np.argmin(sums / counts)]
    )

    true_values = np.array([
        FUNCTIONS[fname](x) for x in Xs
    ])

    selected = float(true_values[selected_idx])
    oracle = float(np.min(true_values))

    return {
        "selected_regret": selected,
        "oracle_regret": oracle,
        "selection_gap": selected - oracle,
        "search_evals": n_search,
        "validation_evals": n_val,
        "total_evals": BUDGET
    }

# ------------------------------------------------------------
# Mandatory reproduction check:
# popsize=8 must match saved Experiment 7
# ------------------------------------------------------------

print("CHECKING EXPERIMENT 7 REPRODUCTION")

for fname, noise, seed in [
    ("Sphere", 0.0, 42),
    ("Rosenbrock", 0.05, 123),
    ("Rastrigin", 0.05, 42)
]:

    X, y, _ = corrected_cma_run(
        fname, noise, seed, 8
    )

    path = (
        ROOT / "experiment_7/search"
        / f"{fname}_CMA-ES_n{noise:.3f}"
          f"_s{seed}_B600.npz"
    )

    with np.load(path, allow_pickle=False) as saved:
        old_X = saved["X"]
        old_y = saved["y"]

    np.testing.assert_allclose(
        X, old_X, rtol=1e-10, atol=1e-10
    )
    np.testing.assert_allclose(
        y, old_y, rtol=1e-10, atol=1e-10
    )

    print("PASS:", fname, noise, seed)

print("DEFAULT CMA-ES REPRODUCTION PASSED")

# ------------------------------------------------------------
# Run corrected sensitivity experiment
# ------------------------------------------------------------

records = []
total = (
    len(FUNCTION_NAMES)
    * len(NOISE_LEVELS)
    * len(SEEDS)
    * len(POP_SIZES)
)

counter = 0

print("\nSTARTING CORRECTED EXPERIMENT 8A")
print("Search trajectories:", total)

for fname in FUNCTION_NAMES:
    for noise in NOISE_LEVELS:
        for seed in SEEDS:
            for popsize in POP_SIZES:

                counter += 1

                X, y, runtime = corrected_cma_run(
                    fname, noise, seed, popsize
                )

                for fraction in FRACTIONS:

                    result = corrected_policy(
                        X, y, fname, noise,
                        seed, fraction
                    )

                    records.append({
                        "function": fname,
                        "noise": noise,
                        "seed": seed,
                        "popsize": popsize,
                        "fraction": fraction,
                        "runtime": runtime,
                        **result
                    })

                if counter % 40 == 0:
                    print(
                        f"Completed {counter}/{total}",
                        flush=True
                    )

df = pd.DataFrame(records)

assert len(df) == total * len(FRACTIONS)
assert (df["total_evals"] == BUDGET).all()
assert (df["selection_gap"] >= -1e-7).all()

df.to_csv(
    OUT / "results.csv",
    index=False
)

summary = df.groupby(
    ["function", "noise", "popsize", "fraction"]
).agg(
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    mean_gap=("selection_gap", "mean"),
    mean_runtime=("runtime", "mean")
).reset_index()

summary.to_csv(
    OUT / "summary.csv",
    index=False
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2, figsize=(12, 9)
)

for ax, fname in zip(
    axes.flat, FUNCTION_NAMES
):

    subset = summary[
        (summary["function"] == fname)
        & (summary["noise"] == 0.05)
    ]

    for fraction in FRACTIONS:

        s = subset[
            subset["fraction"] == fraction
        ].sort_values("popsize")

        ax.plot(
            s["popsize"],
            s["mean_regret"],
            marker="o",
            label=f"Validation {fraction:.0%}"
        )

    ax.set_title(fname)
    ax.set_xlabel("CMA-ES population size")
    ax.set_ylabel("Mean true simple regret")
    ax.set_xticks(POP_SIZES)
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle(
    "Corrected CMA-ES Population Sensitivity"
)
fig.tight_layout()

fig.savefig(
    OUT / "population_sensitivity.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

metadata = {
    "experiment": "8A-R",
    "budget": BUDGET,
    "population_sizes": POP_SIZES,
    "functions": FUNCTION_NAMES,
    "noise_levels": NOISE_LEVELS,
    "seeds": SEEDS,
    "validation_fractions": FRACTIONS,
    "reference": "Experiment 7 CMA-ES",
    "only_modified_parameter": "popsize"
}

(OUT / "metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("\nCORRECTED EXPERIMENT 8A COMPLETE")
print("Saved to:", OUT)
print("\nNOISY RESULTS:")
print(
    summary[summary["noise"] == 0.05]
    .round(5).to_string(index=False)
)


## Corrected 8B-R (authoritative shortlist-diversity sensitivity)


In [ ]:

# ============================================================
# EXPERIMENT 8B-R
# Corrected shortlist-diversity experiment
#
# Uses EXACT Experiment 7 validation noise.
# Verifies top-noisy baseline against Experiment 7.
# No new optimiser searches.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/DSA4212_Final_Project"
)

SEARCH_DIR = ROOT / "experiment_7/search"
MAIN_RESULTS = ROOT / "experiment_7/all_results.csv"

OUT = ROOT / "experiment_8b_corrected"
OUT.mkdir(parents=True, exist_ok=True)

FUNCTIONS8BR = ["Rosenbrock", "Rastrigin"]
METHODS8BR = ["BO", "CMA-ES"]
SEEDS8BR = [42, 123] + list(range(1000, 1018))

# Exact ordering used by Experiment 7 noise7()
F7_ORDER = [
    "Sphere", "Ackley", "Rosenbrock", "Rastrigin"
]
M7_ORDER = ["BO", "CMA-ES"]

BUDGET = 600
NOISE = 0.05
FRACTIONS = [0.10, 0.25]
K = 5
MIN_DIST = 0.15

assert SEARCH_DIR.exists()
assert MAIN_RESULTS.exists()
assert CFG.d == 5
assert len(SEEDS8BR) == 20

main_df = pd.read_csv(MAIN_RESULTS)

# ------------------------------------------------------------
# 2. EXACT EXPERIMENT 7 VALIDATION NOISE
# ------------------------------------------------------------

def validation_noise_8br(
    fname, method, noise, seed, idx, rep
):
    sequence = np.random.SeedSequence([
        7007,
        F7_ORDER.index(fname),
        M7_ORDER.index(method),
        int(round(noise * 100000)),
        int(seed),
        int(idx),
        int(rep)
    ])

    rng = np.random.default_rng(sequence)

    return float(
        rng.normal(0, noise * SCALES[fname])
    )

# Verify against existing Experiment 7 function
# if it is available in the current runtime.

if "noise7" in globals():

    for fname in FUNCTIONS8BR:
        for method in METHODS8BR:

            a = validation_noise_8br(
                fname, method, 0.05, 42, 17, 2
            )

            b = noise7(
                fname, method, 0.05, 42, 17, 2
            )

            assert a == b

    print("PASS: Validation noise matches Experiment 7")

# ------------------------------------------------------------
# 3. SHORTLIST CONSTRUCTION
# ------------------------------------------------------------

def shortlist_8br(X, y, strategy):

    order = np.argsort(
        y, kind="stable"
    )

    if strategy == "top_noisy":
        return order[:K].astype(int)

    if strategy != "diverse":
        raise ValueError(strategy)

    # Same diversity rule as original Experiment 8B.
    Xn = (X - CFG.low) / (CFG.high - CFG.low)

    selected = []

    for candidate in order:

        idx = int(candidate)

        if not selected:
            selected.append(idx)
            continue

        distances = np.linalg.norm(
            Xn[selected] - Xn[idx],
            axis=1
        )

        if np.min(distances) >= MIN_DIST:
            selected.append(idx)

        if len(selected) == K:
            break

    if len(selected) < K:
        for candidate in order:
            idx = int(candidate)

            if idx not in selected:
                selected.append(idx)

            if len(selected) == K:
                break

    assert len(selected) == K
    assert len(set(selected)) == K

    return np.asarray(selected, dtype=int)

# ------------------------------------------------------------
# 4. EVALUATE VALIDATION POLICY
# ------------------------------------------------------------

def evaluate_8br(
    X, y, fname, method,
    seed, fraction, strategy
):

    n_validation = int(round(BUDGET * fraction))
    n_search = BUDGET - n_validation

    Xs = X[:n_search]
    ys = y[:n_search]

    shortlist = shortlist_8br(
        Xs, ys, strategy
    )

    func = FUNCTIONS[fname]

    sums = ys[shortlist].copy()
    counts = np.ones(K, dtype=int)

    for j in range(n_validation):

        slot = j % K
        idx = int(shortlist[slot])
        rep = int(counts[slot])

        observation = (
            func(Xs[idx])
            + validation_noise_8br(
                fname, method, NOISE,
                seed, idx, rep
            )
        )

        sums[slot] += observation
        counts[slot] += 1

    selected_slot = int(
        np.argmin(sums / counts)
    )

    selected_idx = int(
        shortlist[selected_slot]
    )

    # Offline true-value scoring.
    true_values = np.asarray([
        func(x) for x in Xs
    ])

    oracle_idx = int(np.argmin(true_values))

    oracle = float(
        true_values[oracle_idx]
    )

    shortlist_best = float(
        np.min(true_values[shortlist])
    )

    selected = float(
        true_values[selected_idx]
    )

    shortlist_loss = shortlist_best - oracle
    ranking_loss = selected - shortlist_best

    assert np.isclose(
        selected,
        oracle + shortlist_loss + ranking_loss,
        atol=1e-7
    )

    assert shortlist_loss >= -1e-7
    assert ranking_loss >= -1e-7
    assert n_search + n_validation == BUDGET

    return {
        "selected_idx": selected_idx,
        "selected_regret": selected,
        "oracle_regret": oracle,
        "shortlist_loss": shortlist_loss,
        "ranking_loss": ranking_loss,
        "selection_gap": selected - oracle,
        "coverage": int(
            oracle_idx in shortlist
        ),
        "search_evals": n_search,
        "validation_evals": n_validation,
        "total_evals": BUDGET
    }

# ------------------------------------------------------------
# 5. RUN ALL POLICIES
# ------------------------------------------------------------

records = []

print("\nSTARTING EXPERIMENT 8B-R")
print("Expected policy results: 320")
print("=" * 70)

for fname in FUNCTIONS8BR:
    for method in METHODS8BR:
        for seed in SEEDS8BR:

            filename = (
                f"{fname}_{method}_n{NOISE:.3f}"
                f"_s{seed}_B{BUDGET}.npz"
            )

            path = SEARCH_DIR / filename
            assert path.exists(), path

            with np.load(
                path, allow_pickle=False
            ) as saved:
                X = saved["X"].copy()
                y = saved["y"].copy()

            assert X.shape == (BUDGET, CFG.d)
            assert y.shape == (BUDGET,)
            assert np.isfinite(X).all()
            assert np.isfinite(y).all()

            for fraction in FRACTIONS:
                for strategy in ["top_noisy", "diverse"]:

                    result = evaluate_8br(
                        X, y, fname, method,
                        seed, fraction, strategy
                    )

                    records.append({
                        "function": fname,
                        "method": method,
                        "seed": seed,
                        "noise": NOISE,
                        "fraction": fraction,
                        "strategy": strategy,
                        **result
                    })

df8br = pd.DataFrame(records)

assert len(df8br) == 320
assert not df8br.isna().any().any()
assert (df8br["total_evals"] == BUDGET).all()

# ------------------------------------------------------------
# 6. MANDATORY EXPERIMENT 7 REPRODUCTION AUDIT
# ------------------------------------------------------------

baseline = df8br[
    df8br["strategy"] == "top_noisy"
].copy()

reference = main_df[
    (main_df["function"].isin(FUNCTIONS8BR))
    & (main_df["method"].isin(METHODS8BR))
    & np.isclose(main_df["noise"], NOISE)
    & (main_df["K"] == K)
    & (main_df["fraction"].isin(FRACTIONS))
].copy()

keys = [
    "function", "method",
    "seed", "fraction"
]

assert len(baseline) == 160
assert len(reference) == 160

merged = baseline.merge(
    reference,
    on=keys,
    suffixes=("_8br", "_7"),
    validate="one_to_one"
)

assert len(merged) == 160

for metric in [
    "selected_regret",
    "oracle_regret",
    "selection_gap",
    "selected_idx"
]:

    np.testing.assert_allclose(
        merged[f"{metric}_8br"],
        merged[f"{metric}_7"],
        rtol=1e-10,
        atol=1e-8,
        err_msg=f"Mismatch in {metric}"
    )

print("\nPASS: All 160 top-noisy policies")
print("exactly reproduce Experiment 7 (K=5).")

# ------------------------------------------------------------
# 7. SAVE RESULTS AND SUMMARIES
# ------------------------------------------------------------

df8br.to_csv(
    OUT / "results.csv", index=False
)

summary = df8br.groupby(
    ["function", "method", "fraction", "strategy"]
).agg(
    mean_regret=("selected_regret", "mean"),
    std_regret=("selected_regret", "std"),
    mean_oracle=("oracle_regret", "mean"),
    mean_shortlist_loss=("shortlist_loss", "mean"),
    mean_ranking_loss=("ranking_loss", "mean"),
    coverage=("coverage", "mean")
).reset_index()

summary.to_csv(
    OUT / "summary.csv", index=False
)

print("\nEXPERIMENT 8B-R SUMMARY")
print("=" * 70)
print(summary.round(5).to_string(index=False))

# ------------------------------------------------------------
# 8. PAIRED BOOTSTRAP COMPARISONS
# ------------------------------------------------------------

rng = np.random.default_rng(8123)
comparisons = []

for (fname, method, fraction), group in df8br.groupby(
    ["function", "method", "fraction"]
):

    pivot = group.pivot(
        index="seed",
        columns="strategy",
        values="selected_regret"
    )

    assert len(pivot) == 20
    assert not pivot.isna().any().any()

    diff = (
        pivot["diverse"] - pivot["top_noisy"]
    ).to_numpy()

    boot = rng.choice(
        diff,
        size=(10000, len(diff)),
        replace=True
    ).mean(axis=1)

    lo, hi = np.percentile(
        boot, [2.5, 97.5]
    )

    comparisons.append({
        "function": fname,
        "method": method,
        "fraction": fraction,
        "mean_difference": float(diff.mean()),
        "ci_lower": float(lo),
        "ci_upper": float(hi),
        "n_seeds": len(diff)
    })

comparison = pd.DataFrame(comparisons)

comparison.to_csv(
    OUT / "paired_comparisons.csv",
    index=False
)

print("\nPAIRED COMPARISONS")
print("=" * 70)
print(comparison.round(5).to_string(index=False))

# ------------------------------------------------------------
# 9. SAVE FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2, figsize=(12, 5)
)

for ax, fname in zip(axes, FUNCTIONS8BR):

    subset = summary[
        summary["function"] == fname
    ]

    for method in METHODS8BR:
        for strategy in ["top_noisy", "diverse"]:

            s = subset[
                (subset["method"] == method)
                & (subset["strategy"] == strategy)
            ].sort_values("fraction")

            ax.plot(
                s["fraction"] * 100,
                s["mean_regret"],
                marker="o",
                label=f"{method}, {strategy}"
            )

    ax.set_title(fname)
    ax.set_xlabel("Validation allocation (%)")
    ax.set_ylabel("Mean final true regret")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle(
    "Experiment 8B-R: Shortlist Diversity"
)
fig.tight_layout()

fig.savefig(
    OUT / "diversity_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. SAVE METADATA
# ------------------------------------------------------------

metadata = {
    "experiment": "8B-R",
    "reference": "Experiment 7",
    "budget": BUDGET,
    "noise_fraction": NOISE,
    "functions": FUNCTIONS8BR,
    "methods": METHODS8BR,
    "seeds": SEEDS8BR,
    "fractions": FRACTIONS,
    "shortlist_size": K,
    "minimum_normalized_distance": MIN_DIST,
    "validation_noise_seed": 7007,
    "baseline_reproduction": "160/160 passed",
    "note": (
        "Reuses Experiment 7 search trajectories "
        "and validation noise. Diversity strategy "
        "is a post-hoc mechanism investigation."
    )
}

(OUT / "metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("\nEXPERIMENT 8B-R COMPLETE")
print("Saved to:", OUT)
print("Baseline reproduction: PASS")
